In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#1. 데이터 생성

In [ ]:
!pip install -q google-generativeai

In [ ]:
import json
import random
import time
import re
import google.generativeai as genai
from collections import Counter

GEMINI_API_KEY = ""
genai.configure(api_key=GEMINI_API_KEY)
model_gemini = genai.GenerativeModel("gemini-flash-latest")

NOTICES_PATH = '/content/drive/MyDrive/data_all.json'
OUTPUT_PATH  = '/content/drive/MyDrive/synthetic_clicks_v2.json'

with open(NOTICES_PATH, encoding='utf-8') as f:
    notices = json.load(f)

print(f"공지 총 {len(notices)}건 로드 완료")

# ── 트랙제 단과대 ─────────────────────────────────────────────
TRACK_COLLEGE_MAP = {
    "IT공과대학": ["모바일소프트웨어트랙", "빅데이터트랙", "디지털콘텐츠ㆍ가상현실트랙", "웹공학트랙", "전자트랙", "시스템반도체트랙", "기계시스템디자인트랙", "AI로봇융합트랙", "산업공학트랙", "응용산업데이터공학트랙"],
    "디자인대학": ["패션마케팅트랙", "패션디자인트랙", "패션크리에이티브디렉션트랙", "미디어디자인트랙", "시각디자인트랙", "영상ㆍ애니메이션디자인트랙", "UX/UI디자인트랙", "인테리어디자인트랙", "VMDㆍ전시디자인트랙", "게임그래픽디자인트랙", "뷰티디자인매니지먼트학과"],
    "미래융합사회과학대학": ["국제무역트랙", "글로벌비지니스트랙", "기업ㆍ경제분석트랙", "경제금융투자트랙", "공공행정트랙", "법&정책트랙", "부동산트랙", "스마트도시ㆍ교통계획트랙", "기업경영트랙", "비지니스애널리틱스트랙", "회계ㆍ재무경영트랙"],
    "크리에이티브인문예술대학": ["영미문화콘텐츠트랙", "영미언어정보트랙", "한국어교육트랙", "역사문화큐레이션트랙", "역사콘텐츠트랙", "지식정보문화트랙", "디지털인문정보학트랙", "동양화전공", "서양화전공", "한국무용전공", "현대무용전공", "발레전공"],
}

# ── 학과 입학 단과대 ──────────────────────────────────────────
DEPT_COLLEGE_MAP = {
    "창의융합대학": ["상상력인재학부", "문학문화콘텐츠학과", "AI응용학과", "융합보안학과", "미래모빌리티학과"],
    "글로벌인재대학": ["한국언어문화교육학과", "글로벌K비지니스학과", "영상엔터테인먼트학과", "패션뷰티크리에이션학과", "SW융합학과", "글로벌벤처창업학과"],
    "미래플러스대학": ["융합행정학과", "호텔외식경영학과", "뷰티디자인학과", "비지니스컨설팅학과", "ICT융합디자인학과", "AIㆍ소프트웨어학과", "뷰티매니지먼트학과", "디지털콘텐츠디자인학과", "인테리어디자인학과", "스마트제조혁신컨설팅학과"],
}

# ── 트랙별 전공 설명 ──────────────────────────────────────────
TRACK_DESC = {
    # IT공과대학
    "모바일소프트웨어트랙": "iOS/Android 앱 개발, 모바일 서비스 개발",
    "빅데이터트랙": "데이터 분석, 머신러닝, 통계",
    "디지털콘텐츠ㆍ가상현실트랙": "VR/AR, 디지털 콘텐츠 제작",
    "웹공학트랙": "웹 개발, 프론트엔드/백엔드",
    "전자트랙": "전자공학, 회로설계, 임베디드",
    "시스템반도체트랙": "반도체 설계, 칩 개발",
    "기계시스템디자인트랙": "기계공학, 제품설계, CAD",
    "AI로봇융합트랙": "인공지능, 로봇공학, 자율주행",
    "산업공학트랙": "생산관리, 물류, 경영공학",
    "응용산업데이터공학트랙": "산업 데이터 분석, 스마트팩토리",
    # 디자인대학
    "패션마케팅트랙": "패션 비즈니스, 마케팅, 브랜딩",
    "패션디자인트랙": "의상 디자인, 패턴, 소재",
    "패션크리에이티브디렉션트랙": "패션 크리에이티브, 스타일링",
    "미디어디자인트랙": "미디어 아트, 영상 디자인",
    "시각디자인트랙": "그래픽 디자인, 브랜드 아이덴티티",
    "영상ㆍ애니메이션디자인트랙": "영상 제작, 애니메이션",
    "UX/UI디자인트랙": "UX/UI 설계, 사용자 경험",
    "인테리어디자인트랙": "공간 디자인, 인테리어",
    "VMDㆍ전시디자인트랙": "전시 기획, 공간 연출",
    "게임그래픽디자인트랙": "게임 그래픽, 캐릭터 디자인",
    "뷰티디자인매니지먼트학과": "뷰티 산업, 화장품, 메이크업",
    # 미래융합사회과학대학
    "국제무역트랙": "무역, 수출입, 글로벌 비즈니스",
    "글로벌비지니스트랙": "글로벌 경영, 해외 비즈니스",
    "기업ㆍ경제분석트랙": "기업 분석, 경제 리서치",
    "경제금융투자트랙": "금융, 투자, 자산관리",
    "공공행정트랙": "행정, 공무원, 공공정책",
    "법&정책트랙": "법학, 정책, 규제",
    "부동산트랙": "부동산, 도시개발",
    "스마트도시ㆍ교통계획트랙": "도시계획, 교통, 스마트시티",
    "기업경영트랙": "경영, 마케팅, 조직관리",
    "비지니스애널리틱스트랙": "비즈니스 데이터 분석, 경영과학",
    "회계ㆍ재무경영트랙": "회계, 재무, 세무",
    # 크리에이티브인문예술대학
    "영미문화콘텐츠트랙": "영미 문화, 콘텐츠 기획",
    "영미언어정보트랙": "영어학, 언어 정보처리",
    "한국어교육트랙": "한국어 교육, 외국인 대상 교육",
    "역사문화큐레이션트랙": "역사, 문화유산, 박물관 큐레이션",
    "역사콘텐츠트랙": "역사 콘텐츠, 문화 기획",
    "지식정보문화트랙": "지식관리, 정보문화",
    "디지털인문정보학트랙": "디지털 인문학, 데이터 인문학",
    "동양화전공": "동양화, 수묵화, 전통 미술",
    "서양화전공": "서양화, 유화, 현대 미술",
    "한국무용전공": "한국 전통무용, 무용 공연",
    "현대무용전공": "현대무용, 창작무용",
    "발레전공": "발레, 클래식 무용",
    # 창의융합대학
    "상상력인재학부": "전공 자유 선택, 융합적 사고",
    "문학문화콘텐츠학과": "문학, 문화 콘텐츠 기획",
    "AI응용학과": "AI 응용, 인공지능 서비스",
    "융합보안학과": "정보보안, 사이버보안",
    "미래모빌리티학과": "자율주행, 전기차, 미래 교통",
    # 글로벌인재대학
    "한국언어문화교육학과": "한국어 교육, 한국 문화",
    "글로벌K비지니스학과": "한국 비즈니스, 글로벌 경영",
    "영상엔터테인먼트학과": "K-콘텐츠, 영상 엔터테인먼트",
    "패션뷰티크리에이션학과": "K-패션, K-뷰티",
    "SW융합학과": "소프트웨어, IT 융합",
    "글로벌벤처창업학과": "글로벌 창업, 벤처",
    # 미래플러스대학
    "융합행정학과": "행정, 공공관리",
    "호텔외식경영학과": "호텔, 외식, 관광 경영",
    "뷰티디자인학과": "뷰티, 헤어, 메이크업",
    "비지니스컨설팅학과": "경영 컨설팅, 비즈니스 전략",
    "ICT융합디자인학과": "ICT, 디지털 디자인",
    "AIㆍ소프트웨어학과": "AI, 소프트웨어 개발",
    "뷰티매니지먼트학과": "뷰티 산업 경영",
    "디지털콘텐츠디자인학과": "디지털 콘텐츠, 미디어",
    "인테리어디자인학과": "인테리어, 공간 디자인",
    "스마트제조혁신컨설팅학과": "스마트 제조, 생산 혁신",
}

# ── 전체 트랙 (상상력인재학부용) ─────────────────────────────
ALL_TRACKS = [track for tracks in TRACK_COLLEGE_MAP.values() for track in tracks]

# ── 단과대별 가중치 ───────────────────────────────────────────
COLLEGE_WEIGHTS = {
    "IT공과대학":            25,
    "디자인대학":            20,
    "미래융합사회과학대학":   20,
    "크리에이티브인문예술대학": 15,
    "창의융합대학":          15,
    "글로벌인재대학":         5,
    "미래플러스대학":         5,
}

# ── 대학별 특이사항 ───────────────────────────────────────────
COLLEGE_NOTES = {
    "글로벌인재대학": "외국인 유학생 대상 단과대학. 한국 문화/언어, 국제교류, 유학생 지원 프로그램에 관심이 많음. (한국인 대상 일반 취업/채용 공지는 거의 관심 없음)",
    "미래플러스대학": "재직자(직장인) 전형 단과대학. 야간/주말 학사행정, 직무 역량 강화 특강, 자기계발에 관심이 높음. (이미 직장이 있으므로 일반적인 신입 취업/인턴십 공지, 평일 주간 행사는 참여 불가하므로 무시함)",
}

YEARS = ["1학년", "2학년", "3학년", "4학년"]

# ── 학년별 관심사 풀 ──────────────────────────────────────────
YEAR_CATEGORIES = {
    "1학년": ["학사행정", "장학금", "비교과", "봉사/서포터즈", "교육/특강", "공모전/경진대회", "ROTC", "국제교류", "기숙사/생활관"],
    "2학년": ["학사행정", "장학금", "비교과", "봉사/서포터즈", "교육/특강", "공모전/경진대회", "ROTC", "국제교류"],
    "3학년": ["취업/채용", "학사행정", "장학금", "비교과", "봉사/서포터즈", "교육/특강", "공모전/경진대회", "창업", "국제교류"],
    "4학년": ["취업/채용", "학사행정", "장학금", "교육/특강", "공모전/경진대회", "창업"],
}

# ── 페르소나 800개 생성 (중복 제거) ──────────────────────────
random.seed(42)
colleges = list(COLLEGE_WEIGHTS.keys())
weights  = list(COLLEGE_WEIGHTS.values())

personas = []
seen     = set()
max_try  = 10000
try_count = 0

while len(personas) < 800 and try_count < max_try:
    try_count += 1
    college = random.choices(colleges, weights=weights, k=1)[0]
    year    = random.choice(YEARS)

    # 트랙/학과 결정
    if college in TRACK_COLLEGE_MAP:
        if year == "1학년":
            track    = "트랙 미정"
            track_desc = "아직 트랙을 선택하지 않은 1학년"
            year_note  = "1학년으로 아직 트랙을 선택하지 않은 상태. 트랙 안내, 수강신청, 비교과 프로그램에 관심이 높음."
        else:
            track      = random.choice(TRACK_COLLEGE_MAP[college])
            track_desc = TRACK_DESC.get(track, "")
            year_note  = ""

    elif college == "창의융합대학":
        dept = random.choice(DEPT_COLLEGE_MAP[college])
        if dept == "상상력인재학부":
            if year == "1학년":
                track      = "상상력인재학부 (트랙 미정)"
                track_desc = "전체 트랙 중 자유롭게 선택 가능한 융합 학부"
                year_note  = "상상력인재학부 1학년으로 아직 트랙 선택 전."
            else:
                selected   = random.choice(ALL_TRACKS)
                track      = f"상상력인재학부 ({selected} 선택)"
                track_desc = TRACK_DESC.get(selected, "융합적 관심사")
                year_note  = ""
        else:
            track      = dept
            track_desc = TRACK_DESC.get(dept, "")
            year_note  = ""
    else:
        track      = random.choice(DEPT_COLLEGE_MAP[college])
        track_desc = TRACK_DESC.get(track, "")
        year_note  = ""

    # 관심사 결정
    if college == "글로벌인재대학":
        interests = ["국제교류", "비교과"]
        if year in ["3학년", "4학년"]:
            interests.append("학사행정")
    elif college == "미래플러스대학":
        interests = ["학사행정", "교육/특강"]
        if year in ["3학년", "4학년"]:
            interests.append("취업/채용")
    else:
        available_cats = YEAR_CATEGORIES.get(year, list(YEAR_CATEGORIES["3학년"]))
        interests = random.sample(available_cats, random.randint(2, 3))

    # 상상력인재학부 트랙 미정이면 학사행정 필수
    if "트랙 미정" in track and "학사행정" not in interests:
        interests.append("학사행정")

    # 중복 체크
    key = f"{college}_{track}_{year}_{tuple(sorted(interests))}"
    if key in seen:
        continue
    seen.add(key)

    base_note = COLLEGE_NOTES.get(
        college,
        "일반 학부생. 자신의 전공 및 관심사 위주로 공지를 확인하며, 본인 학년에서 현실적으로 참여 가능한 프로그램 위주로 클릭함."
    )
    note = f"{base_note} {year_note}".strip()

    personas.append({
        "id":         f"U{len(personas):03d}",
        "college":    college,
        "track":      track,
        "track_desc": track_desc,
        "year":       year,
        "interests":  interests,
        "note":       note,
    })

print(f"페르소나 {len(personas)}개 생성 완료 (시도: {try_count}회)")

# 분포 확인
from collections import Counter
print("\n단과대별 분포:")
dist = Counter(p['college'] for p in personas)
for college, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {college}: {count}명")

print("\n학년별 분포:")
year_dist = Counter(p['year'] for p in personas)
for year, count in sorted(year_dist.items()):
    print(f"  {year}: {count}명")

# ── 공지 샘플링 함수 ──────────────────────────────────────────
def sample_notices_for_persona(persona, notices, n=50):
    interests = persona['interests']

    # 관심사 카테고리 공지 30개
    interest_notices = [n for n in notices if n['category'] in interests]
    interest_sample = random.sample(interest_notices, min(30, len(interest_notices)))

    # 나머지 20개는 랜덤
    other_notices = [n for n in notices if n['category'] not in interests]
    other_sample = random.sample(other_notices, min(20, len(other_notices)))

    sampled = interest_sample + other_sample
    random.shuffle(sampled)
    return sampled

# ── Gemini API 호출 함수 ──────────────────────────────────────
def generate_clicks(persona, sampled_notices):
    notice_list = [
        {
            "id":       n['id'],
            "category": n['category'],
            "title":    n['title'],
            "body":     n.get('body', '')[:100]
        }
        for n in sampled_notices
    ]

    prompt = f"""
너는 대학 공지사항 추천 시스템의 학습 데이터를 생성하는 AI 시뮬레이터야.
반드시 JSON 형식으로만 답해. 다른 말 하지 마.

[학생 페르소나]
- 소속: {persona['college']} {persona['track']}
- 전공 특성: {persona['track_desc']}
- 학년: {persona['year']}
- 관심사: {', '.join(persona['interests'])}
- 특이사항: {persona['note']}

[공지사항 리스트]
{json.dumps(notice_list, ensure_ascii=False)}

[작업 지시]
위 학생의 전공 특성과 관심사를 바탕으로 판단해.
- Positive: 관심사 관련 공지 + 전공과 관련된 공지 + 현실적으로 참여할 법한 공지 (전공과 전혀 무관한 타 전공생 전용 공지는 절대 클릭하지 않음)
- Hard Negative: 관련 있어 보이지만 전공/관심사/학년 조건이 맞지 않아 무시할 공지

1. positive: 클릭할 공지 ID 8개
2. hard_negative: 무시할 공지 ID 8개

반드시 아래 JSON 형식으로만 출력해:
{{"positive": ["N0001", ...], "hard_negative": ["N0003", ...]}}
"""

    try:
        response = model_gemini.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'```json|```', '', text).strip()
        result = json.loads(text)
        return result
    except Exception as e:
        print(f"  오류: {e}")
        time.sleep(5)
        return None

# ── 메인 실행 ─────────────────────────────────────────────────
synthetic_dataset = []
failed = 0

try:
    with open(OUTPUT_PATH, encoding='utf-8') as f:
        synthetic_dataset = json.load(f)
    processed_ids = {d['user']['id'] for d in synthetic_dataset}
    print(f"\n기존 {len(synthetic_dataset)}개 로드 -> 이어서 실행")
except:
    processed_ids = set()
    print("\n처음부터 시작")

total = len(personas)
for i, persona in enumerate(personas):
    if persona['id'] in processed_ids:
        print(f"[{i+1}/{total}] 스킵: {persona['id']}")
        continue

    sampled = sample_notices_for_persona(persona, notices)
    result  = generate_clicks(persona, sampled)

    if result:
        synthetic_dataset.append({
            "user":               persona,
            "clicks":             result,
            "sampled_notice_ids": [n['id'] for n in sampled]
        })
        print(f"[{i+1}/{total}] {persona['id']} | {persona['college']} {persona['track']} {persona['year']} | 관심사: {persona['interests']} | positive: {len(result.get('positive', []))}개 hard_negative: {len(result.get('hard_negative', []))}개")
    else:
        failed += 1
        print(f"[{i+1}/{total}] {persona['id']} 실패")

    if (i + 1) % 50 == 0:
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(synthetic_dataset, f, ensure_ascii=False, indent=2)
        print(f"-- 중간 저장 완료 ({i+1}/{total}) --")

    time.sleep(4)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(synthetic_dataset, f, ensure_ascii=False, indent=2)

print(f"\n완료! {len(synthetic_dataset)}개 생성 | 실패: {failed}개")
print(f"저장: {OUTPUT_PATH}")

공지 총 1921건 로드 완료
페르소나 800개 생성 완료 (시도: 909회)

단과대별 분포:
  IT공과대학: 175명
  디자인대학: 173명
  미래융합사회과학대학: 141명
  크리에이티브인문예술대학: 134명
  창의융합대학: 128명
  미래플러스대학: 29명
  글로벌인재대학: 20명

학년별 분포:
  1학년: 157명
  2학년: 204명
  3학년: 219명
  4학년: 220명

기존 748개 로드 -> 이어서 실행
[1/800] 스킵: U000
[2/800] 스킵: U001
[3/800] 스킵: U002
[4/800] 스킵: U003
[5/800] 스킵: U004
[6/800] 스킵: U005
[7/800] 스킵: U006
[8/800] 스킵: U007
[9/800] 스킵: U008
[10/800] 스킵: U009
[11/800] 스킵: U010
[12/800] 스킵: U011
[13/800] 스킵: U012
[14/800] 스킵: U013
[15/800] 스킵: U014
[16/800] 스킵: U015
[17/800] 스킵: U016
[18/800] 스킵: U017
[19/800] 스킵: U018
[20/800] 스킵: U019
[21/800] 스킵: U020
[22/800] 스킵: U021
[23/800] 스킵: U022
[24/800] 스킵: U023
[25/800] 스킵: U024
[26/800] 스킵: U025
[27/800] 스킵: U026
[28/800] 스킵: U027
[29/800] 스킵: U028
[30/800] 스킵: U029
[31/800] 스킵: U030
[32/800] 스킵: U031
[33/800] 스킵: U032
[34/800] 스킵: U033
[35/800] 스킵: U034
[36/800] 스킵: U035
[37/800] 스킵: U036
[38/800] 스킵: U037
[39/800] 스킵: U038
[40/800] 스킵: U039
[41/800] 스킵: U040
[42/800] 스킵: U041
[43/80

## 아래에서 데이터 분포를 확인해본 결과, class imblance가 존재하여 데이터 보강하기로 결정함.

In [ ]:
import json
from collections import Counter

with open('/content/drive/MyDrive/synthetic_clicks_v2.json', encoding='utf-8') as f:
    clicks = json.load(f)

with open('/content/drive/MyDrive/data_all.json', encoding='utf-8') as f:
    notices = json.load(f)

notice_map = {n['id']: n for n in notices}

positive_cats = []
negative_cats = []

for d in clicks:
    for nid in d['clicks'].get('positive', []):
        if nid in notice_map:
            positive_cats.append(notice_map[nid]['category'])
    for nid in d['clicks'].get('hard_negative', []):
        if nid in notice_map:
            negative_cats.append(notice_map[nid]['category'])

pos_counter = Counter(positive_cats)
neg_counter = Counter(negative_cats)
all_cats = sorted(set(positive_cats + negative_cats))

print(f"총 페르소나: {len(clicks)}개")
print(f"총 Positive: {len(positive_cats)}건")
print(f"총 Negative: {len(negative_cats)}건")
print()
print(f"{'카테고리':<20} {'Positive':>10} {'Negative':>10} {'P비율':>8} {'N비율':>8}")
print("-" * 65)

for cat in all_cats:
    pos   = pos_counter.get(cat, 0)
    neg   = neg_counter.get(cat, 0)
    total = pos + neg
    pos_ratio = pos / total * 100 if total > 0 else 0
    neg_ratio = neg / total * 100 if total > 0 else 0

    print(f"{cat:<20} {pos:>10} {neg:>10} {pos_ratio:>7.1f}% {neg_ratio:>7.1f}%{flag}")

총 페르소나: 800개
총 Positive: 6400건
총 Negative: 6400건

카테고리                   Positive   Negative      P비율      N비율
-----------------------------------------------------------------
ROTC                        164        268    38.0%    62.0%
공모전/경진대회                    516        403    56.1%    43.9%
교육/특강                       531        578    47.9%    52.1%
국제교류                        469         54    89.7%    10.3%
기숙사/생활관                      61         21    74.4%    25.6%
기타                           20          5    80.0%    20.0%
봉사/서포터즈                     501        133    79.0%    21.0%
비교과                         768        275    73.6%    26.4%
인턴십                         196        291    40.2%    59.8%
장학금                         654        479    57.7%    42.3%
창업                          358         90    79.9%    20.1%
취업/채용                       892       2282    28.1%    71.9%
학사행정                       1220       1483    45.1%    54.9%
학자금/근로장학                     5

#2. 데이터 보강

In [ ]:
import json
import random
import time
import re
import google.generativeai as genai

GEMINI_API_KEY = ""
genai.configure(api_key=GEMINI_API_KEY)
model_gemini = genai.GenerativeModel("gemini-flash-latest")

NOTICES_PATH  = '/content/drive/MyDrive/data_all.json'
BOOST_OUTPUT  = '/content/drive/MyDrive/synthetic_clicks_boost_v2.json'

with open(NOTICES_PATH, encoding='utf-8') as f:
    notices = json.load(f)

notice_map = {n['id']: n for n in notices}
cat_notices = {}
for n in notices:
    cat = n['category']
    if cat not in cat_notices:
        cat_notices[cat] = []
    cat_notices[cat].append(n)

print(f"공지 {len(notices)}건 로드 완료")

# ── 트랙별 설명 ───────────────────────────────────────────────
TRACK_DESC = {
    "모바일소프트웨어트랙": "iOS/Android 앱 개발, 모바일 서비스 개발",
    "빅데이터트랙": "데이터 분석, 머신러닝, 통계",
    "디지털콘텐츠ㆍ가상현실트랙": "VR/AR, 디지털 콘텐츠 제작",
    "웹공학트랙": "웹 개발, 프론트엔드/백엔드",
    "전자트랙": "전자공학, 회로설계, 임베디드",
    "시스템반도체트랙": "반도체 설계, 칩 개발",
    "기계시스템디자인트랙": "기계공학, 제품설계, CAD",
    "AI로봇융합트랙": "인공지능, 로봇공학, 자율주행",
    "산업공학트랙": "생산관리, 물류, 경영공학",
    "응용산업데이터공학트랙": "산업 데이터 분석, 스마트팩토리",
    "패션마케팅트랙": "패션 비즈니스, 마케팅, 브랜딩",
    "패션디자인트랙": "의상 디자인, 패턴, 소재",
    "패션크리에이티브디렉션트랙": "패션 크리에이티브, 스타일링",
    "미디어디자인트랙": "미디어 아트, 영상 디자인",
    "시각디자인트랙": "그래픽 디자인, 브랜드 아이덴티티",
    "영상ㆍ애니메이션디자인트랙": "영상 제작, 애니메이션",
    "UX/UI디자인트랙": "UX/UI 설계, 사용자 경험",
    "인테리어디자인트랙": "공간 디자인, 인테리어",
    "VMDㆍ전시디자인트랙": "전시 기획, 공간 연출",
    "게임그래픽디자인트랙": "게임 그래픽, 캐릭터 디자인",
    "뷰티디자인매니지먼트학과": "뷰티 산업, 화장품, 메이크업",
    "국제무역트랙": "무역, 수출입, 글로벌 비즈니스",
    "글로벌비지니스트랙": "글로벌 경영, 해외 비즈니스",
    "기업ㆍ경제분석트랙": "기업 분석, 경제 리서치",
    "경제금융투자트랙": "금융, 투자, 자산관리",
    "공공행정트랙": "행정, 공무원, 공공정책",
    "법&정책트랙": "법학, 정책, 규제",
    "부동산트랙": "부동산, 도시개발",
    "스마트도시ㆍ교통계획트랙": "도시계획, 교통, 스마트시티",
    "기업경영트랙": "경영, 마케팅, 조직관리",
    "비지니스애널리틱스트랙": "비즈니스 데이터 분석, 경영과학",
    "회계ㆍ재무경영트랙": "회계, 재무, 세무",
    "영미문화콘텐츠트랙": "영미 문화, 콘텐츠 기획",
    "영미언어정보트랙": "영어학, 언어 정보처리",
    "한국어교육트랙": "한국어 교육, 외국인 대상 교육",
    "역사문화큐레이션트랙": "역사, 문화유산, 박물관 큐레이션",
    "역사콘텐츠트랙": "역사 콘텐츠, 문화 기획",
    "지식정보문화트랙": "지식관리, 정보문화",
    "디지털인문정보학트랙": "디지털 인문학, 데이터 인문학",
    "동양화전공": "동양화, 수묵화, 전통 미술",
    "서양화전공": "서양화, 유화, 현대 미술",
    "한국무용전공": "한국 전통무용, 무용 공연",
    "현대무용전공": "현대무용, 창작무용",
    "발레전공": "발레, 클래식 무용",
    "AI응용학과": "AI 응용, 인공지능 서비스",
    "융합보안학과": "정보보안, 사이버보안",
    "미래모빌리티학과": "자율주행, 전기차, 미래 교통",
    "문학문화콘텐츠학과": "문학, 문화 콘텐츠 기획",
    "AIㆍ소프트웨어학과": "AI, 소프트웨어 개발",
    "비지니스컨설팅학과": "경영 컨설팅, 비즈니스 전략",
}

# ── 보강 페르소나 설정 ────────────────────────────────────────
BOOST_PERSONA_MAP = {
    "국제교류": {
        "personas": [
            {
                "college": "IT공과대학", "track": "웹공학트랙",
                "year": "4학년", "interests": ["취업/채용", "공모전/경진대회"],
                "note": "해외에 나갈 생각이 전혀 없는 국내파 학생. 취업 준비에만 집중하며 교환학생이나 어학연수 등 국제교류 프로그램에는 1%의 관심도 없음."
            },
            {
                "college": "미래융합사회과학대학", "track": "공공행정트랙",
                "year": "3학년", "interests": ["취업/채용", "학사행정"],
                "note": "현재 공무원 시험 준비에 올인하고 있어 국제교류 프로그램에는 전혀 관심 없음."
            },
            {
                "college": "IT공과대학", "track": "시스템반도체트랙",
                "year": "4학년", "interests": ["취업/채용", "장학금"],
                "note": "대기업 반도체 취업만 목표로 하며 해외 교환학생 프로그램에는 관심 없음."
            },
            {
                "college": "미래융합사회과학대학", "track": "회계ㆍ재무경영트랙",
                "year": "4학년", "interests": ["취업/채용", "창업"],
                "note": "회계사 시험 준비 중으로 국제교류 프로그램 참여 여력 전혀 없음."
            },
        ],
        "n_each": 15
    },
    "창업": {
        "personas": [
            {
                "college": "크리에이티브인문예술대학", "track": "발레전공",
                "year": "2학년", "interests": ["비교과", "봉사/서포터즈"],
                "note": "오직 안정적인 취업만을 목표로 하며 스타트업이나 창업처럼 위험 부담이 있는 활동은 절대 피하고 싶어 하는 보수적인 성향."
            },
            {
                "college": "글로벌인재대학", "track": "한국언어문화교육학과",
                "year": "3학년", "interests": ["국제교류", "장학금"],
                "note": "안정적인 직장을 원하며 창업 리스크를 극도로 기피하는 학생."
            },
            {
                "college": "크리에이티브인문예술대학", "track": "한국어교육트랙",
                "year": "2학년", "interests": ["학사행정", "비교과"],
                "note": "교원 임용고시 준비 중으로 창업 관련 활동에는 전혀 관심 없음."
            },
        ],
        "n_each": 10
    },
    "봉사/서포터즈": {
        "personas": [
            {
                "college": "IT공과대학", "track": "빅데이터트랙",
                "year": "4학년", "interests": ["취업/채용", "공모전/경진대회"],
                "note": "이미 졸업에 필요한 봉사시간을 모두 채운 고학년. 개인 취업 스펙 쌓기에도 시간이 부족하여 봉사나 서포터즈 활동을 할 여력이 전혀 없음."
            },
            {
                "college": "미래융합사회과학대학", "track": "경제금융투자트랙",
                "year": "3학년", "interests": ["취업/채용", "장학금"],
                "note": "금융권 취업 준비에 올인 중으로 봉사/서포터즈 활동에는 시간을 쓰기 싫어하는 학생."
            },
            {
                "college": "IT공과대학", "track": "모바일소프트웨어트랙",
                "year": "4학년", "interests": ["취업/채용", "교육/특강"],
                "note": "개발 포트폴리오 준비에만 집중하며 봉사활동에는 관심 없는 학생."
            },
        ],
        "n_each": 10
    },
    "비교과": {
        "personas": [
            {
                "college": "IT공과대학", "track": "AI로봇융합트랙",
                "year": "3학년", "interests": ["취업/채용", "공모전/경진대회"],
                "note": "정규 전공 학점 관리에만 목숨을 거는 학생. 학점으로 인정되지 않는 비교과 프로그램은 시간 낭비라고 생각함."
            },
            {
                "college": "미래융합사회과학대학", "track": "비지니스애널리틱스트랙",
                "year": "4학년", "interests": ["취업/채용", "장학금"],
                "note": "자격증 시험과 취업 준비로 바빠서 비교과 활동에 참여할 시간이 없는 학생."
            },
            {
                "college": "디자인대학", "track": "패션디자인트랙",
                "year": "3학년", "interests": ["취업/채용", "창업"],
                "note": "포트폴리오 제작에만 집중하며 수료증 위주의 비교과 프로그램에는 관심 없음."
            },
        ],
        "n_each": 10
    },
    "기숙사/생활관": {
        "personas": [
            {
                "college": "IT공과대학", "track": "웹공학트랙",
                "year": "2학년", "interests": ["비교과", "공모전/경진대회"],
                "note": "학교 근처 본가에서 부모님과 함께 거주하며 지하철로 통학. 기숙사 입사 공지에는 전혀 관심 없음."
            },
            {
                "college": "미래융합사회과학대학", "track": "기업경영트랙",
                "year": "3학년", "interests": ["취업/채용", "장학금"],
                "note": "이미 자취방 계약을 완료하여 기숙사나 생활관 입사 공지에는 전혀 관심 없는 학생."
            },
        ],
        "n_each": 10
    },
}

# ── 보강 페르소나 생성 ────────────────────────────────────────
random.seed(77)
boost_personas = []

for boost_cat, config in BOOST_PERSONA_MAP.items():
    for persona_template in config['personas']:
        for i in range(config['n_each']):
            boost_personas.append({
                "id":         f"BOOST_{boost_cat}_{len(boost_personas):03d}",
                "college":    persona_template['college'],
                "track":      persona_template['track'],
                "track_desc": TRACK_DESC.get(persona_template['track'], ""),
                "year":       persona_template['year'],
                "interests":  persona_template['interests'],
                "note":       persona_template['note'],
                "boost_cat":  boost_cat,
            })

print(f"보강 페르소나 총 {len(boost_personas)}개 생성")
for cat in BOOST_PERSONA_MAP:
    count = sum(1 for p in boost_personas if p['boost_cat'] == cat)
    print(f"  {cat}: {count}개")

# ── 보강용 공지 샘플링 ────────────────────────────────────────
def sample_boost_notices(boost_cat, notices, n=50):
    # 타겟 카테고리 20개 강제 포함
    target_notices = cat_notices.get(boost_cat, [])
    target_sample  = random.sample(target_notices, min(20, len(target_notices)))

    # 나머지 30개 랜덤
    other_notices = [n for n in notices if n['category'] != boost_cat]
    other_sample  = random.sample(other_notices, min(30, len(other_notices)))

    sampled = target_sample + other_sample
    random.shuffle(sampled)
    return sampled

# ── Gemini API 호출 ───────────────────────────────────────────
def generate_boost_clicks(persona, sampled_notices):
    notice_list = [
        {
            "id":       n['id'],
            "category": n['category'],
            "title":    n['title'],
            "body":     n.get('body', '')[:100]
        }
        for n in sampled_notices
    ]

    boost_cat = persona['boost_cat']

    prompt = f"""
너는 대학 공지사항 추천 시스템의 학습 데이터를 생성하는 AI 시뮬레이터야.
반드시 JSON 형식으로만 답해. 다른 말 하지 마.

[학생 페르소나]
- 소속: {persona['college']} {persona['track']}
- 전공 특성: {persona['track_desc']}
- 학년: {persona['year']}
- 관심사: {', '.join(persona['interests'])}
- 특이사항: {persona['note']}
★ 특별 지시: 이 학생은 [{boost_cat}] 카테고리에 전혀 관심이 없으며, 참여할 수 없는 상황입니다.

[공지사항 리스트]
{json.dumps(notice_list, ensure_ascii=False)}

[작업 지시]
위 학생의 전공 특성과 관심사를 바탕으로 판단해.
- Positive: 관심사 관련 공지 + 전공과 관련된 공지 (전공과 전혀 무관한 공지는 절대 클릭하지 않음)
- Hard Negative: 관련 있어 보이지만 전공/관심사/학년 조건이 맞지 않아 무시할 공지

★ 중요: [{boost_cat}] 카테고리의 공지가 목록에 있다면, 반드시 최소 3개 이상을 hard_negative 리스트에 포함시켜야 합니다!!

1. positive: 클릭할 공지 ID 8개
2. hard_negative: 무시할 공지 ID 8개 (반드시 [{boost_cat}] 카테고리 공지 최소 3개 포함)

반드시 아래 JSON 형식으로만 출력해:
{{"positive": ["N0001", ...], "hard_negative": ["N0003", ...]}}
"""

    try:
        response = model_gemini.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'```json|```', '', text).strip()
        result = json.loads(text)
        return result
    except Exception as e:
        print(f"  오류: {e}")
        time.sleep(5)
        return None

# ── 메인 실행 ─────────────────────────────────────────────────
boost_dataset = []
failed = 0

try:
    with open(BOOST_OUTPUT, encoding='utf-8') as f:
        boost_dataset = json.load(f)
    processed_ids = {d['user']['id'] for d in boost_dataset}
    print(f"\n기존 {len(boost_dataset)}개 로드 -> 이어서 실행")
except:
    processed_ids = set()
    print("\n처음부터 시작")

total = len(boost_personas)
for i, persona in enumerate(boost_personas):
    if persona['id'] in processed_ids:
        print(f"[{i+1}/{total}] 스킵: {persona['id']}")
        continue

    sampled = sample_boost_notices(persona['boost_cat'], notices)
    result  = generate_boost_clicks(persona, sampled)

    if result:
        # boost_cat 공지가 실제로 hard_negative에 포함됐는지 확인
        neg_boost = [nid for nid in result.get('hard_negative', [])
                     if notice_map.get(nid, {}).get('category') == persona['boost_cat']]

        boost_dataset.append({
            "user":               persona,
            "clicks":             result,
            "sampled_notice_ids": [n['id'] for n in sampled]
        })
        print(f"[{i+1}/{total}] {persona['id']} | {persona['boost_cat']} | boost_neg: {len(neg_boost)}개")
    else:
        failed += 1
        print(f"[{i+1}/{total}] {persona['id']} 실패")

    if (i + 1) % 20 == 0:
        with open(BOOST_OUTPUT, 'w', encoding='utf-8') as f:
            json.dump(boost_dataset, f, ensure_ascii=False, indent=2)
        print(f"-- 중간 저장 완료 ({i+1}/{total}) --")

    time.sleep(4)

with open(BOOST_OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(boost_dataset, f, ensure_ascii=False, indent=2)

print(f"\n완료! {len(boost_dataset)}개 생성 | 실패: {failed}개")

# ── 합친 후 분포 확인 ─────────────────────────────────────────
from collections import Counter

with open('/content/drive/MyDrive/synthetic_clicks_v2.json', encoding='utf-8') as f:
    original = json.load(f)

combined = original + boost_dataset
print(f"\n기존: {len(original)}개 | 보강: {len(boost_dataset)}개 | 합계: {len(combined)}개")

positive_cats = []
negative_cats = []
for d in combined:
    for nid in d['clicks'].get('positive', []):
        if nid in notice_map:
            positive_cats.append(notice_map[nid]['category'])
    for nid in d['clicks'].get('hard_negative', []):
        if nid in notice_map:
            negative_cats.append(notice_map[nid]['category'])

pos_counter = Counter(positive_cats)
neg_counter = Counter(negative_cats)
all_cats    = sorted(set(positive_cats + negative_cats))

print(f"\n{'카테고리':<20} {'Positive':>10} {'Negative':>10} {'N비율':>8}")
print("-" * 55)
for cat in all_cats:
    pos   = pos_counter.get(cat, 0)
    neg   = neg_counter.get(cat, 0)
    total = pos + neg
    neg_ratio = neg / total * 100 if total > 0 else 0
    flag = " ← 보강됨!" if cat in BOOST_PERSONA_MAP else ""
    print(f"{cat:<20} {pos:>10} {neg:>10} {neg_ratio:>7.1f}%{flag}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


공지 1921건 로드 완료
보강 페르소나 총 170개 생성
  국제교류: 60개
  창업: 30개
  봉사/서포터즈: 30개
  비교과: 30개
  기숙사/생활관: 20개

처음부터 시작
[1/170] BOOST_국제교류_000 | 국제교류 | boost_neg: 3개
[2/170] BOOST_국제교류_001 | 국제교류 | boost_neg: 6개
[3/170] BOOST_국제교류_002 | 국제교류 | boost_neg: 6개
[4/170] BOOST_국제교류_003 | 국제교류 | boost_neg: 4개
[5/170] BOOST_국제교류_004 | 국제교류 | boost_neg: 3개
[6/170] BOOST_국제교류_005 | 국제교류 | boost_neg: 5개
[7/170] BOOST_국제교류_006 | 국제교류 | boost_neg: 4개
[8/170] BOOST_국제교류_007 | 국제교류 | boost_neg: 3개
[9/170] BOOST_국제교류_008 | 국제교류 | boost_neg: 4개
[10/170] BOOST_국제교류_009 | 국제교류 | boost_neg: 3개
[11/170] BOOST_국제교류_010 | 국제교류 | boost_neg: 4개
[12/170] BOOST_국제교류_011 | 국제교류 | boost_neg: 4개
[13/170] BOOST_국제교류_012 | 국제교류 | boost_neg: 3개
[14/170] BOOST_국제교류_013 | 국제교류 | boost_neg: 3개
[15/170] BOOST_국제교류_014 | 국제교류 | boost_neg: 4개
[16/170] BOOST_국제교류_015 | 국제교류 | boost_neg: 4개
[17/170] BOOST_국제교류_016 | 국제교류 | boost_neg: 6개
[18/170] BOOST_국제교류_017 | 국제교류 | boost_neg: 4개
[19/170] BOOST_국제교류_018 | 국제교류 | boost_neg: 4개
[20/170] BO

In [ ]:
import json

with open('/content/drive/MyDrive/synthetic_clicks_v2.json', encoding='utf-8') as f:
    original = json.load(f)

with open('/content/drive/MyDrive/synthetic_clicks_boost_v2.json', encoding='utf-8') as f:
    boost = json.load(f)

combined = original + boost
print(f"기존: {len(original)}개 | 보강: {len(boost)}개 | 합계: {len(combined)}개")

with open('/content/drive/MyDrive/data_clicks_combined.json', 'w', encoding='utf-8') as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print("저장 완료!")

기존: 800개 | 보강: 170개 | 합계: 970개
저장 완료!


#3. 모델 학습

In [ ]:
import json
import torch
import numpy as np
import random
from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

NOTICES_PATH    = '/content/drive/MyDrive/data_all.json'
CLICKS_PATH     = '/content/drive/MyDrive/data_clicks_combined.json'
MODEL_SAVE_PATH = '/content/drive/MyDrive/two_tower_model_v3.pt'

with open(NOTICES_PATH, encoding='utf-8') as f:
    notices = json.load(f)

with open(CLICKS_PATH, encoding='utf-8') as f:
    clicks = json.load(f)

print(f"공지 {len(notices)}건 | 클릭 데이터 {len(clicks)}개 로드 완료")

# ── SBERT 임베딩 ──────────────────────────────────────────────
print("\nSBERT 로드 중...")
sbert = SentenceTransformer('jhgan/ko-sroberta-multitask')
sbert = sbert.to(device)

print("공지 임베딩 생성 중...")
notice_texts = [
    f"{n['category']} {n['title']} {n.get('body', '')[:100]}"
    for n in notices
]
notice_embeddings = sbert.encode(
    notice_texts, batch_size=64,
    show_progress_bar=True, device=device, convert_to_numpy=True
)
print(f"공지 임베딩 shape: {notice_embeddings.shape}")

scores = np.array([n.get('notice_score', 0) for n in notices])
max_score = scores.max() if scores.max() > 0 else 1
scores_normalized = scores / max_score
notice_id2idx = {n['id']: i for i, n in enumerate(notices)}

# ── 유저 임베딩 ───────────────────────────────────────────────
print("\n유저 임베딩 중...")
user_texts = []
for user_data in clicks:
    info = user_data['user']
    user_texts.append(
        f"{info['college']} {info.get('track', '')} {info['year']} 관심사: {', '.join(info['interests'])}"
    )

user_texts_unique = list(set(user_texts))
user_emb_dict = dict(zip(
    user_texts_unique,
    sbert.encode(user_texts_unique, convert_to_numpy=True, show_progress_bar=True)
))
print(f"유저 텍스트 {len(user_texts_unique)}개 임베딩 완료")

# ── 유저 단위 Split (sorted로 재현성 보장) ────────────────────
random.seed(42)
unique_user_ids = sorted(list(set(d['user']['id'] for d in clicks)))  # sorted 추가!
random.shuffle(unique_user_ids)

train_end = int(len(unique_user_ids) * 0.7)
val_end   = int(len(unique_user_ids) * 0.85)

train_ids = set(unique_user_ids[:train_end])
val_ids   = set(unique_user_ids[train_end:val_end])
test_ids  = set(unique_user_ids[val_end:])

train_clicks = [d for d in clicks if d['user']['id'] in train_ids]
val_clicks   = [d for d in clicks if d['user']['id'] in val_ids]
test_clicks  = [d for d in clicks if d['user']['id'] in test_ids]

print(f"\n유저 단위 Split:")
print(f"  Train: {len(train_clicks)}명 | Val: {len(val_clicks)}명 | Test: {len(test_clicks)}명")

# ── Dataset ───────────────────────────────────────────────────
class TwoTowerDataset(Dataset):
    def __init__(self, clicks_data, notice_id2idx, notice_embeddings, scores_normalized, user_emb_dict):
        self.samples           = []
        self.notice_embeddings = notice_embeddings
        self.scores_normalized = scores_normalized

        for user_data in clicks_data:
            info = user_data['user']
            user_text = f"{info['college']} {info.get('track', '')} {info['year']} 관심사: {', '.join(info['interests'])}"
            user_emb = user_emb_dict[user_text]

            # Positive: 유저당 1개만 랜덤 샘플링
            positives = [
                notice_id2idx[pos_id]
                for pos_id in user_data['clicks'].get('positive', [])
                if pos_id in notice_id2idx
            ]
            if positives:
                pos_idx = random.choice(positives)
                self.samples.append({
                    'user_emb':   user_emb,
                    'notice_idx': pos_idx,
                    'label':      1.0,
                })

            # Hard Negative: 최대 3개 샘플링 (1:3 비율)
            negatives = [
                notice_id2idx[neg_id]
                for neg_id in user_data['clicks'].get('hard_negative', [])
                if neg_id in notice_id2idx
            ]
            if negatives:
                sampled_negs = random.sample(negatives, min(3, len(negatives)))
                for neg_idx in sampled_negs:
                    self.samples.append({
                        'user_emb':   user_emb,
                        'notice_idx': neg_idx,
                        'label':      -1.0,
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        n_idx  = sample['notice_idx']
        item_emb = np.concatenate([
            self.notice_embeddings[n_idx],
            [self.scores_normalized[n_idx]]
        ])
        return {
            'user_emb': torch.tensor(sample['user_emb'], dtype=torch.float),
            'item_emb': torch.tensor(item_emb,           dtype=torch.float),
            'label':    torch.tensor(sample['label'],     dtype=torch.float),
        }

print("\n데이터셋 생성 중...")
train_dataset = TwoTowerDataset(train_clicks, notice_id2idx, notice_embeddings, scores_normalized, user_emb_dict)
val_dataset   = TwoTowerDataset(val_clicks,   notice_id2idx, notice_embeddings, scores_normalized, user_emb_dict)
test_dataset  = TwoTowerDataset(test_clicks,  notice_id2idx, notice_embeddings, scores_normalized, user_emb_dict)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, drop_last=False)

print(f"Train: {len(train_dataset)}개 | Val: {len(val_dataset)}개 | Test: {len(test_dataset)}개")

# ── 모델 ──────────────────────────────────────────────────────
EMBED_DIM  = notice_embeddings.shape[1]  # 768
HIDDEN_DIM = 256
OUTPUT_DIM = 128

class UserTower(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return F.normalize(self.fc(x), dim=-1)

class ItemTower(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim + 1, hidden_dim),  # +1: notice_score
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return F.normalize(self.fc(x), dim=-1)

class TwoTowerModel(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.user_tower = UserTower(embed_dim, hidden_dim, output_dim)
        self.item_tower = ItemTower(embed_dim, hidden_dim, output_dim)
    def forward(self, user_emb, item_emb):
        return self.user_tower(user_emb), self.item_tower(item_emb)

model = TwoTowerModel(EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
print(f"\n모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# ── 하이브리드 Loss ───────────────────────────────────────────
TEMPERATURE      = 0.05
INFO_WEIGHT      = 1.0
COSINE_WEIGHT    = 0.2
criterion_cosine = nn.CosineEmbeddingLoss(margin=0.2).to(device)

def combined_loss(user_vecs, item_vecs, labels):
    loss_cosine = criterion_cosine(user_vecs, item_vecs, labels)

    pos_mask = labels == 1.0
    if pos_mask.sum() >= 2:
        pos_user  = user_vecs[pos_mask]
        pos_item  = item_vecs[pos_mask]
        logits    = torch.matmul(pos_user, pos_item.T) / TEMPERATURE
        target    = torch.arange(pos_user.size(0), device=user_vecs.device)
        loss_info = F.cross_entropy(logits, target)
    else:
        loss_info = torch.tensor(0.0, device=user_vecs.device)

    return INFO_WEIGHT * loss_info + COSINE_WEIGHT * loss_cosine, loss_info, loss_cosine

# ── 학습 ─────────────────────────────────────────────────────
EPOCHS        = 20
LEARNING_RATE = 3e-4
PATIENCE      = 3

optimizer     = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
best_val_loss = float('inf')
best_epoch    = 0
patience_cnt  = 0

print("\n학습 시작! (InfoNCE × 1.0 + CosineEmbeddingLoss × 0.2)")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    total_info_loss  = 0.0
    total_cos_loss   = 0.0
    train_steps      = 0

    for batch in train_loader:
        user_emb = batch['user_emb'].to(device)
        item_emb = batch['item_emb'].to(device)
        label    = batch['label'].to(device)

        optimizer.zero_grad()
        user_vec, item_vec = model(user_emb, item_emb)
        loss, loss_info, loss_cosine = combined_loss(user_vec, item_vec, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()
        total_info_loss  += loss_info.item()
        total_cos_loss   += loss_cosine.item()
        train_steps      += 1

    avg_train_loss = total_train_loss / max(train_steps, 1)
    avg_info_loss  = total_info_loss  / max(train_steps, 1)
    avg_cos_loss   = total_cos_loss   / max(train_steps, 1)

    model.eval()
    total_val_loss = 0.0
    val_steps      = 0

    with torch.no_grad():
        for batch in val_loader:
            user_emb = batch['user_emb'].to(device)
            item_emb = batch['item_emb'].to(device)
            label    = batch['label'].to(device)

            user_vec, item_vec = model(user_emb, item_emb)
            loss, _, _ = combined_loss(user_vec, item_vec, label)
            total_val_loss += loss.item()
            val_steps      += 1

    avg_val_loss = total_val_loss / max(val_steps, 1)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch    = epoch + 1
        patience_cnt  = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        save_mark = " <- Best 저장!"
    else:
        patience_cnt += 1
        save_mark = ""

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {avg_train_loss:.4f} (InfoNCE: {avg_info_loss:.4f} Cosine: {avg_cos_loss:.4f}) | Val: {avg_val_loss:.4f}{save_mark}")

    if patience_cnt >= PATIENCE:
        print(f"\nEarly Stopping! (patience={PATIENCE})")
        break

print(f"\n완료! Best Epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
print(f"저장: {MODEL_SAVE_PATH}")

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
print("Best 모델 로드 완료!")

사용 장치: cuda
공지 1921건 | 클릭 데이터 970개 로드 완료

SBERT 로드 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


공지 임베딩 생성 중...


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

공지 임베딩 shape: (1921, 768)

유저 임베딩 중...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

유저 텍스트 814개 임베딩 완료

유저 단위 Split:
  Train: 679명 | Val: 145명 | Test: 146명

데이터셋 생성 중...
Train: 2716개 | Val: 580개 | Test: 584개

모델 파라미터 수: 459,776

학습 시작! (InfoNCE × 1.0 + CosineEmbeddingLoss × 0.2)
Epoch [01/20] | Train: 4.3930 (InfoNCE: 4.3473 Cosine: 0.2285) | Val: 3.6641 <- Best 저장!
Epoch [02/20] | Train: 3.8941 (InfoNCE: 3.8498 Cosine: 0.2212) | Val: 3.5028 <- Best 저장!
Epoch [03/20] | Train: 3.6809 (InfoNCE: 3.6394 Cosine: 0.2075) | Val: 3.4729 <- Best 저장!
Epoch [04/20] | Train: 3.4664 (InfoNCE: 3.4258 Cosine: 0.2030) | Val: 3.4757
Epoch [05/20] | Train: 3.1685 (InfoNCE: 3.1289 Cosine: 0.1980) | Val: 3.5606
Epoch [06/20] | Train: 2.9153 (InfoNCE: 2.8765 Cosine: 0.1940) | Val: 3.7027

Early Stopping! (patience=3)

완료! Best Epoch: 3 | Best Val Loss: 3.4729
저장: /content/drive/MyDrive/two_tower_model_v3.pt
Best 모델 로드 완료!


## 평가지표 점수는 아래에서 확인가능하다.

In [ ]:
import numpy as np
import torch
from collections import defaultdict

MODEL_WEIGHT = 0.8
SCORE_WEIGHT = 0.1
PROFILE_WEIGHT = 0.1

def profile_match_score(user_info, notice):
    score = 0.0

    college = user_info.get("college", "")
    track = user_info.get("track", "")
    year = str(user_info.get("year", ""))
    note = user_info.get("note", "")

    text = f"{notice.get('category', '')} {notice.get('title', '')} {notice.get('body', '')}"

    if college and college in text:
        score += 0.3
    if track and track != "트랙 미정" and track in text:
        score += 0.3
    if year and year in text:
        score += 0.2

    for word in note.split():
        if len(word) >= 3 and word in text:
            score += 0.02

    return min(score, 1.0)

def evaluate_category_filtered_ensemble(clicks_data, K_list=[10, 30, 50]):
    user_positives = defaultdict(list)
    user_embeddings = {}
    user_interests = {}
    user_infos = {}

    for user_data in clicks_data:
        info = user_data["user"]
        user_text = f"{info['college']} {info.get('track', '')} {info['year']} 관심사: {', '.join(info['interests'])}"

        if user_text not in user_emb_dict:
            continue

        for pos_id in user_data["clicks"].get("positive", []):
            if pos_id in notice_id2idx:
                user_positives[user_text].append(notice_id2idx[pos_id])
                user_embeddings[user_text] = user_emb_dict[user_text]
                user_interests[user_text] = info.get("interests", [])
                user_infos[user_text] = info

    hit_results = {k: [] for k in K_list}
    candidate_sizes = []
    filtered_positive_counts = []

    with torch.no_grad():
        for uid, pos_indices in user_positives.items():
            interests = user_interests[uid]
            user_info = user_infos[uid]

            category_candidate_idx = [
                i for i, n in enumerate(notices)
                if n.get("category", "") in interests
            ]

            if len(category_candidate_idx) == 0:
                continue

            filtered_pos = set(pos_indices) & set(category_candidate_idx)

            if len(filtered_pos) == 0:
                continue

            candidate_sizes.append(len(category_candidate_idx))
            filtered_positive_counts.append(len(filtered_pos))

            user_tensor = torch.tensor(
                user_embeddings[uid],
                dtype=torch.float
            ).unsqueeze(0).to(device)

            user_vec = model.user_tower(user_tensor).cpu().numpy()

            cand_embs = all_item_embs[category_candidate_idx]
            model_scores = np.dot(cand_embs, user_vec.T).flatten()
            notice_scores = scores_normalized[category_candidate_idx]

            profile_scores = np.array([
                profile_match_score(user_info, notices[i])
                for i in category_candidate_idx
            ])

            final_scores = (
                MODEL_WEIGHT * model_scores
                + SCORE_WEIGHT * notice_scores
                + PROFILE_WEIGHT * profile_scores
            )

            ranked_local = np.argsort(final_scores)[::-1]
            ranked_idx = [category_candidate_idx[i] for i in ranked_local]

            for K in K_list:
                top_k = set(ranked_idx[:K])
                hit_count = len(filtered_pos & top_k)
                hit_results[K].append(1.0 if hit_count > 0 else 0.0)

    print("\n=== Category Filtered + Score Ensemble HitRate Evaluation ===")
    print(f"전체 공지 수: {len(notices)}개")
    print(f"평균 후보 공지 수: {np.mean(candidate_sizes):.0f}개")
    print(f"평균 평가 positive 수: {np.mean(filtered_positive_counts):.2f}개")
    print(f"평가 유저 수: {len(hit_results[K_list[0]])}명")
    print(f"평가 positive 수: {sum(filtered_positive_counts)}개")
    print(f"Weight: model={MODEL_WEIGHT}, score={SCORE_WEIGHT}, profile={PROFILE_WEIGHT}")
    print("-" * 55)
    print(f"{'지표':<15} {'@10':>10} {'@30':>10} {'@50':>10}")
    print("-" * 55)

    values = [
        np.mean(hit_results[k]) if len(hit_results[k]) > 0 else 0.0
        for k in K_list
    ]

    print(f"{'HitRate':<15} {values[0]:>10.4f} {values[1]:>10.4f} {values[2]:>10.4f}")
    print("-" * 55)

    return {
        f"HitRate@{k}": float(np.mean(hit_results[k])) if len(hit_results[k]) > 0 else 0.0
        for k in K_list
    }

ensemble_summary = evaluate_category_filtered_ensemble(test_clicks)


=== Category Filtered + Score Ensemble HitRate Evaluation ===
전체 공지 수: 1921개
평균 후보 공지 수: 493개
평균 평가 positive 수: 6.50개
평가 유저 수: 133명
평가 positive 수: 864개
Weight: model=0.8, score=0.1, profile=0.1
-------------------------------------------------------
지표                     @10        @30        @50
-------------------------------------------------------
HitRate             0.1429     0.3233     0.5038
-------------------------------------------------------


#4. 1차 테스트

In [ ]:
import random

ALPHA = 0.85
BETA  = 0.15
INTEREST_MULTIPLIER = 1.3
PENALTY_MULTIPLIER  = 0.5
PENALTY_CATEGORIES  = ['국제교류', '봉사/서포터즈', '창업']
MIN_SCORE_THRESHOLD = 0.05

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

# 모든 공지 Item 벡터 미리 계산 (category onehot 포함)
print("공지 벡터 미리 계산 중...")
all_item_embs = []
with torch.no_grad():
    for i in range(len(notices)):
        item_emb = np.concatenate([
            notice_embeddings[i],
            [scores_normalized[i]],
            category_onehots[i]  # 추가
        ])
        item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)
        item_vec = model.item_tower(item_tensor)
        all_item_embs.append(item_vec.cpu().numpy())

all_item_embs = np.vstack(all_item_embs)
print(f"공지 벡터 계산 완료: {all_item_embs.shape}")


# ── 새 공지 → 유저 추천 함수 ─────────────────────────────────
def recommend_users_for_new_notice(title, category, body="", top_k=5):
    body_preview = body[:100] if body else ""
    notice_text  = f"{category} {title} {body_preview}".strip()
    notice_emb   = sbert.encode([notice_text], convert_to_numpy=True)

    # category onehot 생성
    onehot = np.zeros(len(CATEGORIES), dtype=np.float32)
    if category in cat2idx:
        onehot[cat2idx[category]] = 1.0

    item_emb    = np.concatenate([notice_emb[0], [1.0], onehot])
    item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)

    with torch.no_grad():
        item_vec = model.item_tower(item_tensor).cpu().numpy()

    test_users = [
        {"college": "IT공과대학",             "track": "AI로봇융합트랙",      "year": "1학년", "interests": ["비교과", "학사행정"]},
        {"college": "IT공과대학",             "track": "빅데이터트랙",        "year": "3학년", "interests": ["취업/채용", "공모전/경진대회"]},
        {"college": "IT공과대학",             "track": "모바일소프트웨어트랙", "year": "4학년", "interests": ["취업/채용", "장학금"]},
        {"college": "디자인대학",             "track": "UX/UI디자인트랙",     "year": "2학년", "interests": ["비교과", "봉사/서포터즈"]},
        {"college": "미래융합사회과학대학",    "track": "기업경영트랙",        "year": "4학년", "interests": ["취업/채용", "창업"]},
        {"college": "크리에이티브인문예술대학","track": "영미언어정보트랙",     "year": "2학년", "interests": ["국제교류", "장학금"]},
        {"college": "글로벌인재대학",          "track": "한국언어문화교육학과", "year": "2학년", "interests": ["국제교류", "비교과"]},
        {"college": "미래플러스대학",          "track": "AIㆍ소프트웨어학과",  "year": "3학년", "interests": ["학사행정", "교육/특강"]},
        {"college": "창의융합대학",            "track": "AI응용학과",          "year": "1학년", "interests": ["비교과", "학사행정"]},
        {"college": "IT공과대학",             "track": "웹공학트랙",          "year": "2학년", "interests": ["ROTC", "학사행정"]},
        {"college": "창의융합대학",            "track": "융합보안학과",        "year": "1학년", "interests": ["기숙사/생활관", "장학금"]},
        {"college": "디자인대학",             "track": "패션디자인트랙",      "year": "3학년", "interests": ["봉사/서포터즈", "창업"]},
    ]

    user_scores = []
    with torch.no_grad():
        for user in test_users:
            user_text   = f"{user['college']} {user['track']} {user['year']} 관심사: {', '.join(user['interests'])}"
            user_emb    = sbert.encode([user_text], convert_to_numpy=True)
            user_tensor = torch.tensor(user_emb, dtype=torch.float).to(device)
            user_vec    = model.user_tower(user_tensor).cpu().numpy()

            base_sim      = float(np.dot(item_vec, user_vec.T).flatten()[0])
            stretched_sim = (base_sim + 1.0) / 2.0

            if category in user['interests']:
                final_score = stretched_sim * INTEREST_MULTIPLIER
            elif category in PENALTY_CATEGORIES:
                final_score = stretched_sim * PENALTY_MULTIPLIER
            else:
                final_score = stretched_sim

            user_scores.append({
                'user':        user,
                'base_sim':    base_sim,
                'final_score': final_score,
            })

    user_scores.sort(key=lambda x: x['final_score'], reverse=True)

    print(f"\n{'='*60}")
    print(f"새 공지: [{category}] {title}")
    if body_preview:
        print(f"본문: {body_preview[:50]}...")
    print(f"{'='*60}")
    for rank, res in enumerate(user_scores[:top_k]):
        u = res['user']
        bonus_mark = " ★" if category in u['interests'] else ""
        print(f"  {rank+1}. {u['college']} {u['track']} {u['year']} | 관심사: {', '.join(u['interests'])}{bonus_mark}")
        print(f"     유사도: {res['base_sim']:.4f} | 최종: {res['final_score']:.4f}")


# ── 알림 발송 대상자 선정 함수 ────────────────────────────────
def get_notification_targets(title, category, body="", min_score=0.3):
    body_preview = body[:100] if body else ""
    notice_text  = f"{category} {title} {body_preview}".strip()
    notice_emb   = sbert.encode([notice_text], convert_to_numpy=True)

    onehot = np.zeros(len(CATEGORIES), dtype=np.float32)
    if category in cat2idx:
        onehot[cat2idx[category]] = 1.0

    item_emb = np.concatenate([notice_emb[0], [1.0], onehot])
    item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)

    with torch.no_grad():
        item_vec = model.item_tower(item_tensor).cpu().numpy()

    all_users = [d['user'] for d in clicks]

    temp_scores = []

    with torch.no_grad():
        for user in all_users:
            user_text = f"{user['college']} {user.get('track', '')} {user['year']} 관심사: {', '.join(user['interests'])}"
            user_emb = sbert.encode([user_text], convert_to_numpy=True)
            user_tensor = torch.tensor(user_emb, dtype=torch.float).to(device)
            user_vec = model.user_tower(user_tensor).cpu().numpy()

            base_sim = float(np.dot(item_vec, user_vec.T).flatten()[0])

            temp_scores.append({
                'user': user,
                'base_sim': base_sim
            })

    base_scores = np.array([x['base_sim'] for x in temp_scores])
    min_sim = base_scores.min()
    max_sim = base_scores.max()

    normalized_scores = (base_scores - min_sim) / (max_sim - min_sim + 1e-8)

    user_scores = []

    for idx, item in enumerate(temp_scores):
        user = item['user']

        stretched_sim = normalized_scores[idx] ** 2.5

        if category in user['interests']:
            final_score = stretched_sim * INTEREST_MULTIPLIER
        elif category in PENALTY_CATEGORIES:
            final_score = stretched_sim * PENALTY_MULTIPLIER
        else:
            final_score = stretched_sim

        user_scores.append({
            'user': user,
            'base_sim': item['base_sim'],
            'final_score': final_score
        })

    user_scores.sort(key=lambda x: x['final_score'], reverse=True)

    scores = np.array([u['final_score'] for u in user_scores])
    mean_score = scores.mean()
    std_score = scores.std()
    cut_score = mean_score + std_score

    targets = [u for u in user_scores if u['final_score'] >= max(cut_score, min_score)]

    print(f"\n{'='*60}")
    print(f"새 공지: [{category}] {title}")
    print(f"전체 유저: {len(user_scores)}명 | 알림 대상: {len(targets)}명 ({len(targets)/len(user_scores)*100:.1f}%)")
    print(f"{'='*60}")
    print(f"평균: {mean_score:.4f} | 표준편차: {std_score:.4f} | 컷 기준: {cut_score:.4f}")
    print(f"점수 분포: 최대 {scores.max():.4f} | 최소 {scores.min():.4f}")

    print(f"\n알림 대상 Top 10:")
    for rank, res in enumerate(targets[:10]):
        u = res['user']
        bonus_mark = " ★" if category in u['interests'] else ""
        print(f"  {rank+1}. {u['college']} {u.get('track','')} {u['year']} | 관심사: {', '.join(u['interests'])}{bonus_mark}")
        print(f"     유사도: {res['base_sim']:.4f} | 최종: {res['final_score']:.4f}")

    print(f"\n점수 구간별 유저 수:")
    bins = [1.3, 1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
    for i in range(len(bins)-1):
        count = sum(1 for s in scores if bins[i+1] <= s < bins[i])
        bar = "█" * (count // 5)
        print(f"  {bins[i+1]:.1f}~{bins[i]:.1f}: {count:3d}명 {bar}")

        # ── ❌ 알림 제외된 하위 유저 (Bottom 10) ─────────────────
    import random

    print(f"\n랜덤 유저 샘플 10명:")
    random_users = random.sample(user_scores, min(10, len(user_scores)))

    target_users = set(id(t['user']) for t in targets)
    for rank, res in enumerate(random_users):
        u = res['user']
        mark = " [알림됨]" if id(u) in target_users else " [미알림]"
        interest_mark = " ★" if category in u['interests'] else ""

        print(f"  {rank+1}. {u['college']} {u.get('track','')} {u['year']} | 관심사: {', '.join(u['interests'])}{interest_mark}{mark}")
        print(f"     유사도: {res['base_sim']:.4f} | 최종: {res['final_score']:.4f}")

    return targets


# ── 테스트 ────────────────────────────────────────────────────
_ = recommend_users_for_new_notice(
    title="2026학년도 하반기 삼성전자 채용설명회 안내",
    category="취업/채용",
    body="삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다."
)

_ = recommend_users_for_new_notice(
    title="2026학년도 국가장학금 2차 신청 안내",
    category="장학금",
    body="한국장학재단에서 2026학년도 국가장학금 2차 신청을 받습니다. 소득분위 8구간 이하 재학생이라면 누구나 신청 가능하며, 최대 700만원까지 지원됩니다."
)

_ = recommend_users_for_new_notice(
    title="글로벌 교환학생 파견 프로그램 모집",
    category="국제교류",
    body="2026학년도 2학기 해외 파견 교환학생을 모집합니다. 미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다."
)

_ = get_notification_targets(
    title="2026학년도 하반기 삼성전자 채용설명회 안내",
    category="취업/채용",
    body="삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다.",
)

_ = get_notification_targets(
    title="2026학년도 국가장학금 2차 신청 안내",
    category="장학금",
    body="한국장학재단에서 2026학년도 국가장학금 2차 신청을 받습니다. 소득분위 8구간 이하 재학생이라면 누구나 신청 가능하며, 최대 700만원까지 지원됩니다.",
)

_ = get_notification_targets(
    title="글로벌 교환학생 파견 프로그램 모집",
    category="국제교류",
    body="2026학년도 2학기 해외 파견 교환학생을 모집합니다. 미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다.",
)

공지 벡터 미리 계산 중...
공지 벡터 계산 완료: (1921, 128)

새 공지: [취업/채용] 2026학년도 하반기 삼성전자 채용설명회 안내
본문: 삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다....
  1. 미래융합사회과학대학 기업경영트랙 4학년 | 관심사: 취업/채용, 창업 ★
     유사도: 0.3969 | 최종: 0.9080
  2. IT공과대학 모바일소프트웨어트랙 4학년 | 관심사: 취업/채용, 장학금 ★
     유사도: 0.3537 | 최종: 0.8799
  3. IT공과대학 빅데이터트랙 3학년 | 관심사: 취업/채용, 공모전/경진대회 ★
     유사도: 0.2893 | 최종: 0.8380
  4. 미래플러스대학 AIㆍ소프트웨어학과 3학년 | 관심사: 학사행정, 교육/특강
     유사도: 0.2962 | 최종: 0.6481
  5. 디자인대학 패션디자인트랙 3학년 | 관심사: 봉사/서포터즈, 창업
     유사도: 0.1406 | 최종: 0.5703

새 공지: [장학금] 2026학년도 국가장학금 2차 신청 안내
본문: 한국장학재단에서 2026학년도 국가장학금 2차 신청을 받습니다. 소득분위 8구간 이하 재학...
  1. IT공과대학 모바일소프트웨어트랙 4학년 | 관심사: 취업/채용, 장학금 ★
     유사도: 0.2368 | 최종: 0.8039
  2. 크리에이티브인문예술대학 영미언어정보트랙 2학년 | 관심사: 국제교류, 장학금 ★
     유사도: 0.2125 | 최종: 0.7881
  3. 창의융합대학 융합보안학과 1학년 | 관심사: 기숙사/생활관, 장학금 ★
     유사도: 0.2034 | 최종: 0.7822
  4. 미래융합사회과학대학 기업경영트랙 4학년 | 관심사: 취업/채용, 창업
     유사도: 0.2672 | 최종: 0.6336
  5. 글로벌인재대학 한국언어문화교육학과 2학년 | 관심사: 국제교류, 비교과
     유사도: 0.1860 | 최종: 0.5930

새 공지: [국제교

## 1차 테스트 결과, AI 직군임에도 경영대학 학생에게 추천하는 등 트랙, 직무 등 분야에 대한 이해도가 떨어지는 것을 확인할 수 있다. 따라서 job_type을 추가하여 보정하기로 결정하였다.

#5. job_type 추가

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 데이터 로드
with open('/content/drive/MyDrive/data_all.json', encoding='utf-8') as f:
    notices = json.load(f)

print(f"공지 {len(notices)}건 로드 완료")

# SBERT 로드
print("SBERT 로드 중...")
sbert = SentenceTransformer('jhgan/ko-sroberta-multitask')
print("SBERT 로드 완료!")

공지 1921건 로드 완료
SBERT 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT 로드 완료!


In [ ]:
from collections import Counter
import re

def get_top_keywords(category, top_n=30):
    # 해당 카테고리 공지만 필터링
    cat_notices = [n for n in notices if n.get('category') == category]

    # 제목 + 본문 합치기
    all_text = ""
    for n in cat_notices:
        all_text += n.get('title', '') + " "
        all_text += n.get('body', '')[:200] + " "

    # 불용어 제거
    stopwords = ['안내', '공지', '한성', '대학', '학교', '신청', '모집', '관련',
                 '대상', '참가', '참여', '진행', '운영', '실시', '예정', '안내문',
                 '이상', '이하', '이후', '이전', '및', '등', '의', '를', '이',
                 '가', '은', '는', '으로', '에서', '에', '과', '와', '한성공지',
                 '년', '월', '일', '학년도', '학기', '2025', '2026', '2024']

    # 단어 분리 (2글자 이상)
    words = re.findall(r'[가-힣a-zA-Z]{2,}', all_text)
    words = [w for w in words if w not in stopwords and len(w) >= 2]

    counter = Counter(words)

    print(f"\n[{category}] 공지 {len(cat_notices)}건 - 상위 키워드 {top_n}개:")
    print("-" * 50)
    for word, count in counter.most_common(top_n):
        print(f"  {word}: {count}회")

# job_type 필요한 카테고리만 확인
for cat in ["취업/채용", "비교과", "교육/특강", "공모전/경진대회", "창업", "국제교류"]:
    get_top_keywords(cat)


[취업/채용] 공지 533건 - 상위 키워드 30개:
--------------------------------------------------
  채용: 582회
  지원: 278회
  추천채용: 224회
  채용정보: 218회
  공고: 199회
  한성대학교: 188회
  사업: 131회
  회사: 122회
  교내: 120회
  바랍니다: 117회
  개요: 115회
  신입: 109회
  기업으로: 105회
  적극: 105회
  제품: 105회
  가능: 101회
  있는: 96회
  기업: 92회
  고용: 92회
  학사학위: 88회
  없는: 88회
  기준: 85회
  소지자: 85회
  행정조교: 84회
  경우: 82회
  설립: 81회
  추천: 78회
  https: 77회
  채용분야: 70회
  조교: 70회

[비교과] 공지 207건 - 상위 키워드 30개:
--------------------------------------------------
  비교과: 95회
  한성대학교: 72회
  프로그램: 66회
  많은: 39회
  바랍니다: 35회
  동아리: 35회
  pt: 34회
  재학생: 33회
  안녕하세요: 31회
  ESG센터: 31회
  https: 30회
  kr: 29회
  hansung: 27회
  활동: 26회
  한성대: 26회
  ac: 25회
  기간: 25회
  대학생: 25회
  지급: 24회
  ESG: 23회
  있는: 22회
  비교과포인트: 21회
  창업동아리: 20회
  바로가기: 19회
  School: 19회
  선발: 19회
  진로: 19회
  부탁드립니다: 17회
  까지: 17회
  확인: 17회

[교육/특강] 공지 88건 - 상위 키워드 30개:
--------------------------------------------------
  프로그램: 25회
  위한: 25회
  한성대학교: 19회
  수강생: 18회
  많은: 16회
  진로: 15회
  AI: 14회


In [ ]:
# ================================================================
# SBERT 기반 job_type 멀티 라벨 분류
# ================================================================

import torch
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── job_type 대표 문장 정의 ───────────────────────────────────
JOB_TYPE_SENTENCES = {
    "취업/채용": {
        "IT/개발":       "소프트웨어 개발 프로그래밍 AI 데이터 ICT 시스템 엔지니어 백엔드 프론트엔드 웹 앱 반도체 IT",
        "경영/금융":     "경영 회계 재무 마케팅 영업 금융 은행 증권 보험 캐피탈 투자 무역 물류",
        "디자인/마케팅": "디자인 마케팅 브랜드 콘텐츠 뷰티 패션 크리에이티브 광고 홍보 영상",
        "공공/연구":     "공공기관 연구원 정부 공무원 학술 R&D 재단 진흥원 공단 기관 연구소",
        "교내채용":      "조교 행정 한성대학교 교내 사업단 센터 계약직 임시직 학술연구원",
    },
    "비교과": {
        "IT교육":    "코딩 프로그래밍 소프트웨어 AI 데이터 파이썬 알고리즘 개발 SW",
        "진로/취업": "진로 취업 직무 커리어 멘토링 포트폴리오 자기소개서 면접 직업",
        "디자인":    "디자인 포토샵 영상편집 UX 그래픽 일러스트 미디어 콘텐츠",
        "글쓰기":    "글쓰기 에세이 작문 논문 독서 토론 스피치 발표",
        "심리/상담": "심리 상담 멘탈 스트레스 힐링 마음 정서 치유 감정",
        "ESG/봉사":  "ESG 환경 봉사 사회공헌 탄소 지속가능 서포터즈 동아리",
    },
    "교육/특강": {
        "AI/IT":     "AI 인공지능 TOPCIT 디지털 코딩 소프트웨어 빅데이터 블록체인",
        "진로/취업": "진로 취업 직무 커리어 면접 이력서 직업 멘토링",
        "어학":      "영어 어학 외국어 TOEIC 회화 글로벌 영문 언어",
        "디자인":    "디자인 UX UI 영상 포토샵 그래픽 미디어 콘텐츠",
        "인문/교양": "인문 교양 역사 철학 문화 예술 사회 경제 문학",
        "창업":      "창업 스타트업 아이디어 사업 기업가정신 벤처",
    },
    "공모전/경진대회": {
        "개발/SW":     "SW AI 소프트웨어 개발 코딩 알고리즘 해커톤 프로그래밍 IT",
        "디자인":      "디자인 UX UI 그래픽 영상 포스터 브랜딩 패션 시각 이미지",
        "창업/마케팅": "창업 아이디어 마케팅 비즈니스 스타트업 사업계획서 경영",
        "인문/사회":   "에세이 영어 프레젠테이션 영자신문 스피치 글쓰기 역사 문화",
    },
    "창업": {
        "IT창업":   "AI 기술 소프트웨어 플랫폼 테크 개발 서비스 앱",
        "소셜벤처": "소셜벤처 사회적기업 ESG 임팩트 환경 비영리 공익",
        "일반창업": "예비창업 CEO 아이디어 사업계획 창업지원 아카데미",
    },
    "국제교류": {
        "교환학생":   "교환학생 파견 해외대학 협정 학점인정 유학",
        "해외인턴":   "해외인턴 K-Move 청년봉사단 KOICA 해외현장실습 해외취업",
        "어학연수":   "어학연수 영어권 단기연수 어학캠프 언어",
        "외국인학생": "외국인 유학생 한국어 한국문화 글로벌학생 국제학생",
    },
}

# ── SBERT로 대표 문장 임베딩 ──────────────────────────────────
print("job_type 대표 문장 임베딩 중...")
job_type_embeddings = {}

for category, types in JOB_TYPE_SENTENCES.items():
    job_type_embeddings[category] = {}
    for job_type, sentence in types.items():
        emb = sbert.encode([sentence], convert_to_numpy=True)
        job_type_embeddings[category][job_type] = emb[0]

print("완료!")

# ── 공지 job_type 분류 함수 ───────────────────────────────────
def classify_job_type(notice, threshold=0.3, top_k=2):
    """
    공지를 job_type으로 멀티 라벨 분류
    threshold: 이 유사도 이상이면 해당 job_type 포함
    top_k: 최대 몇 개까지 포함할지
    """
    category = notice.get('category', '')

    # job_type 분류 대상 카테고리인지 확인
    if category not in JOB_TYPE_SENTENCES:
        return []

    # 공지 텍스트 임베딩
    notice_text = f"{notice['title']} {notice.get('body', '')[:100]}"
    notice_emb  = sbert.encode([notice_text], convert_to_numpy=True)

    # 각 job_type과 유사도 계산
    scores = {}
    for job_type, type_emb in job_type_embeddings[category].items():
        sim = cosine_similarity(notice_emb, [type_emb])[0][0]
        scores[job_type] = float(sim)

    # 정렬
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # threshold 이상 + top_k 제한
    result = []
    for job_type, score in sorted_scores[:top_k]:
        if score >= threshold:
            result.append({'job_type': job_type, 'score': score})

    return result

# ── 전체 공지 분류 ────────────────────────────────────────────
print("\n공지 job_type 분류 중...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

# sbert GPU로 이동
sbert = sbert.to(device)

# 카테고리별 대상 공지만 필터링
target_notices = [n for n in notices if n.get('category') in JOB_TYPE_SENTENCES]
print(f"job_type 분류 대상: {len(target_notices)}개")

# GPU로 배치 임베딩
target_texts = [f"{n['title']} {n.get('body', '')[:100]}" for n in target_notices]
target_embs  = sbert.encode(
    target_texts,
    batch_size=128,
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True
)

# job_type 분류
notice_job_types = {}

for i, notice in enumerate(target_notices):
    nid        = notice['id']
    category   = notice['category']
    notice_emb = target_embs[i]

    scores = {}
    for job_type, type_emb in job_type_embeddings[category].items():
        sim = float(np.dot(notice_emb, type_emb) /
                   (np.linalg.norm(notice_emb) * np.linalg.norm(type_emb) + 1e-8))
        scores[job_type] = sim

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result = [
        {'job_type': jt, 'score': s}
        for jt, s in sorted_scores[:2]
        if s >= 0.3
    ]
    notice_job_types[nid] = result

# 규칙 기반 카테고리는 빈 리스트
for notice in notices:
    if notice['id'] not in notice_job_types:
        notice_job_types[notice['id']] = []

print(f"\n분류 완료!")
print(f"job_type 적용: {sum(1 for v in notice_job_types.values() if v)}개")
print(f"규칙 기반: {sum(1 for v in notice_job_types.values() if not v)}개")

job_type 대표 문장 임베딩 중...
완료!

공지 job_type 분류 중...
사용 장치: cuda
job_type 분류 대상: 1003개


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


분류 완료!
job_type 적용: 968개
규칙 기반: 953개


In [ ]:
# 테스트
def test_job_type(title, category, body=""):
    notice = {'title': title, 'category': category, 'body': body, 'id': 'test'}
    result = classify_job_type(notice)

    print(f"\n공지: [{category}] {title}")
    if body:
        print(f"본문: {body[:50]}...")
    if result:
        print(f"job_type:")
        for t in result:
            print(f"  → {t['job_type']} (유사도: {t['score']:.4f})")
    else:
        print("job_type: 분류 불가")

test_job_type(
    title="2026학년도 하반기 삼성전자 채용설명회 안내",
    category="취업/채용",
    body="삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다."
)
test_job_type(
    title="하나은행 신입행원 채용",
    category="취업/채용",
    body="하나은행에서 신입행원을 채용합니다. 경영, 금융, 회계 전공 우대."
)
test_job_type(
    title="한성대학교 교육혁신처 임시직 채용",
    category="취업/채용",
    body="한성대학교 교육혁신처 교육혁신지원센터 임시직 채용 공고입니다. 행정 업무 담당."
)
test_job_type(
    title="AI 활용 디지털 역량 강화 특강",
    category="교육/특강",
    body="생성형 AI 실습 교육 프로그램입니다."
)
test_job_type(
    title="2026학년도 SW 아이디어 경진대회",
    category="공모전/경진대회",
    body="소프트웨어 관련 아이디어 공모전입니다."
)
test_job_type(
    title="글로벌 교환학생 파견 프로그램 모집",
    category="국제교류",
    body="미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다."
)
test_job_type(
    title="2026학년도 국가장학금 2차 신청 안내",
    category="장학금",
    body="소득분위 8구간 이하 재학생 신청 가능."
)


공지: [취업/채용] 2026학년도 하반기 삼성전자 채용설명회 안내
본문: 삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다....
job_type:
  → IT/개발 (유사도: 0.4060)

공지: [취업/채용] 하나은행 신입행원 채용
본문: 하나은행에서 신입행원을 채용합니다. 경영, 금융, 회계 전공 우대....
job_type:
  → 경영/금융 (유사도: 0.3642)
  → 공공/연구 (유사도: 0.3072)

공지: [취업/채용] 한성대학교 교육혁신처 임시직 채용
본문: 한성대학교 교육혁신처 교육혁신지원센터 임시직 채용 공고입니다. 행정 업무 담당....
job_type:
  → 교내채용 (유사도: 0.7478)
  → 공공/연구 (유사도: 0.5202)

공지: [교육/특강] AI 활용 디지털 역량 강화 특강
본문: 생성형 AI 실습 교육 프로그램입니다....
job_type:
  → AI/IT (유사도: 0.6936)
  → 디자인 (유사도: 0.3931)

공지: [공모전/경진대회] 2026학년도 SW 아이디어 경진대회
본문: 소프트웨어 관련 아이디어 공모전입니다....
job_type:
  → 개발/SW (유사도: 0.5597)
  → 창업/마케팅 (유사도: 0.4645)

공지: [국제교류] 글로벌 교환학생 파견 프로그램 모집
본문: 미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다....
job_type:
  → 교환학생 (유사도: 0.6154)
  → 외국인학생 (유사도: 0.4975)

공지: [장학금] 2026학년도 국가장학금 2차 신청 안내
본문: 소득분위 8구간 이하 재학생 신청 가능....
job_type: 분류 불가


In [ ]:
# threshold 0.35로 조정해서 전체 분류
notice_job_types = {}

for i, notice in enumerate(target_notices):
    nid        = notice['id']
    category   = notice['category']
    notice_emb = target_embs[i]

    scores = {}
    for job_type, type_emb in job_type_embeddings[category].items():
        sim = float(np.dot(notice_emb, type_emb) /
                   (np.linalg.norm(notice_emb) * np.linalg.norm(type_emb) + 1e-8))
        scores[job_type] = sim

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result = [
        {'job_type': jt, 'score': s}
        for jt, s in sorted_scores[:2]
        if s >= 0.35
    ]
    notice_job_types[nid] = result

# 규칙 기반 카테고리는 빈 리스트
for notice in notices:
    if notice['id'] not in notice_job_types:
        notice_job_types[notice['id']] = []

print(f"분류 완료!")
print(f"job_type 적용: {sum(1 for v in notice_job_types.values() if v)}개")
print(f"규칙 기반: {sum(1 for v in notice_job_types.values() if not v)}개")

# 카테고리별 분포 확인
from collections import Counter
print(f"\n카테고리별 job_type 분포:")
for cat in JOB_TYPE_SENTENCES.keys():
    cat_notices = [n for n in notices if n.get('category') == cat]
    type_counter = Counter()
    for n in cat_notices:
        for t in notice_job_types.get(n['id'], []):
            type_counter[t['job_type']] += 1
    print(f"\n[{cat}]")
    for jt, count in type_counter.most_common():
        print(f"  {jt}: {count}개")

분류 완료!
job_type 적용: 895개
규칙 기반: 1026개

카테고리별 job_type 분포:

[취업/채용]
  IT/개발: 279개
  교내채용: 262개
  공공/연구: 257개
  경영/금융: 120개
  디자인/마케팅: 11개

[비교과]
  진로/취업: 120개
  ESG/봉사: 82개
  심리/상담: 13개
  IT교육: 13개
  디자인: 11개
  글쓰기: 3개

[교육/특강]
  진로/취업: 69개
  AI/IT: 44개
  인문/교양: 13개
  창업: 7개
  디자인: 5개
  어학: 3개

[공모전/경진대회]
  인문/사회: 26개
  창업/마케팅: 23개
  개발/SW: 22개
  디자인: 19개

[창업]
  일반창업: 41개
  IT창업: 20개
  소셜벤처: 15개

[국제교류]
  교환학생: 37개
  외국인학생: 35개
  해외인턴: 23개
  어학연수: 15개


In [ ]:
# ── 트랙 → 도메인 매핑 ───────────────────────────────────────
TRACK_DOMAIN = {
    "IT": [
        # IT공과대학
        "모바일소프트웨어트랙", "빅데이터트랙", "디지털콘텐츠ㆍ가상현실트랙",
        "웹공학트랙", "전자트랙", "시스템반도체트랙", "기계시스템디자인트랙",
        "AI로봇융합트랙", "산업공학트랙", "응용산업데이터공학트랙",
        # 창의융합대학
        "AI응용학과", "융합보안학과", "미래모빌리티학과",
        # 글로벌인재대학
        "SW융합학과", "글로벌벤처창업학과",
        # 미래플러스대학
        "AIㆍ소프트웨어학과", "ICT융합디자인학과", "스마트제조혁신컨설팅학과",
    ],
    "경영": [
        # 미래융합사회과학대학
        "기업경영트랙", "회계ㆍ재무경영트랙", "경제금융투자트랙",
        "기업ㆍ경제분석트랙", "비지니스애널리틱스트랙",
        "국제무역트랙", "글로벌비지니스트랙",
        # 글로벌인재대학
        "글로벌K비지니스학과",
        # 미래플러스대학
        "비지니스컨설팅학과", "호텔외식경영학과",
    ],
    "행정/공공": [
        # 미래융합사회과학대학
        "공공행정트랙", "법&정책트랙", "부동산트랙",
        "스마트도시ㆍ교통계획트랙",
        # 미래플러스대학
        "융합행정학과",
    ],
    "디자인": [
        # 디자인대학
        "패션마케팅트랙", "패션디자인트랙", "패션크리에이티브디렉션트랙",
        "미디어디자인트랙", "시각디자인트랙", "영상ㆍ애니메이션디자인트랙",
        "UX/UI디자인트랙", "인테리어디자인트랙", "VMDㆍ전시디자인트랙",
        "게임그래픽디자인트랙", "뷰티디자인매니지먼트학과",
        # 글로벌인재대학
        "패션뷰티크리에이션학과", "영상엔터테인먼트학과",
        # 미래플러스대학
        "뷰티디자인학과", "뷰티매니지먼트학과",
        "디지털콘텐츠디자인학과", "인테리어디자인학과",
    ],
    "인문": [
        # 크리에이티브인문예술대학
        "영미문화콘텐츠트랙", "영미언어정보트랙", "한국어교육트랙",
        "역사문화큐레이션트랙", "역사콘텐츠트랙", "지식정보문화트랙",
        "디지털인문정보학트랙",
        # 창의융합대학
        "문학문화콘텐츠학과",
        # 글로벌인재대학
        "한국언어문화교육학과",
    ],
    "예술": [
        # 크리에이티브인문예술대학
        "동양화전공", "서양화전공",
        "한국무용전공", "현대무용전공", "발레전공",
    ],
    "융합": [
        # 창의융합대학
        "상상력인재학부",
    ],
}

# 트랙 → 도메인 역매핑
track_to_domain = {}
for domain, tracks in TRACK_DOMAIN.items():
    for track in tracks:
        track_to_domain[track] = domain

# 트랙 미정은 융합으로
track_to_domain["트랙 미정"] = "융합"
track_to_domain["상상력인재학부 (트랙 미정)"] = "융합"

def get_user_domain(track):
    # 상상력인재학부 (xxx 선택) 처리
    if "상상력인재학부" in track:
        # 선택한 트랙이 있으면 그 도메인으로
        for t, d in track_to_domain.items():
            if t in track and t != "상상력인재학부":
                return d
        return "융합"
    return track_to_domain.get(track, "융합")

# ── 도메인 → 카테고리별 job_type 매핑 ────────────────────────
DOMAIN_TO_JOB_TYPE = {
    "취업/채용": {
        "IT":       "IT/개발",
        "경영":     "경영/금융",
        "행정/공공": "공공/연구",
        "디자인":   "디자인/마케팅",
        "인문":     "공공/연구",
        "예술":     "디자인/마케팅",
        "융합":     None,  # 융합은 매칭 없음
    },
    "비교과": {
        "IT":       "IT교육",
        "경영":     "진로/취업",
        "행정/공공": "진로/취업",
        "디자인":   "디자인",
        "인문":     "글쓰기",
        "예술":     "디자인",
        "융합":     None,
    },
    "교육/특강": {
        "IT":       "AI/IT",
        "경영":     "진로/취업",
        "행정/공공": "인문/교양",
        "디자인":   "디자인",
        "인문":     "인문/교양",
        "예술":     "디자인",
        "융합":     None,
    },
    "공모전/경진대회": {
        "IT":       "개발/SW",
        "경영":     "창업/마케팅",
        "행정/공공": "인문/사회",
        "디자인":   "디자인",
        "인문":     "인문/사회",
        "예술":     "디자인",
        "융합":     None,
    },
    "창업": {
        "IT":       "IT창업",
        "경영":     "일반창업",
        "행정/공공": "소셜벤처",
        "디자인":   "일반창업",
        "인문":     "소셜벤처",
        "예술":     "소셜벤처",
        "융합":     None,
    },
    "국제교류": {
        "IT":       "교환학생",
        "경영":     "교환학생",
        "행정/공공": "교환학생",
        "디자인":   "교환학생",
        "인문":     "어학연수",
        "예술":     "교환학생",
        "융합":     "교환학생",
    },
}

def get_user_job_type(track, category):
    """유저 트랙과 카테고리를 받아서 해당 job_type 반환"""
    domain = get_user_domain(track)
    cat_mapping = DOMAIN_TO_JOB_TYPE.get(category, {})
    return cat_mapping.get(domain, None)

def get_job_type_match_score(track, notice):
    """유저 트랙과 공지의 job_type 매칭 점수 반환"""
    category = notice.get('category', '')
    notice_types = [t['job_type'] for t in notice_job_types.get(notice['id'], [])]

    if not notice_types:
        return 0.0

    user_job_type = get_user_job_type(track, category)

    if user_job_type is None:
        return 0.0

    return 1.0 if user_job_type in notice_types else 0.0

# ── 테스트 ────────────────────────────────────────────────────
print("트랙 → 도메인 매핑 테스트:")
test_tracks = [
    "빅데이터트랙", "기업경영트랙", "UX/UI디자인트랙",
    "영미언어정보트랙", "발레전공", "AI응용학과",
    "공공행정트랙", "상상력인재학부 (웹공학트랙 선택)",
    "트랙 미정"
]
for track in test_tracks:
    domain = get_user_domain(track)
    print(f"  {track} → {domain}")

print(f"\njob_type 매칭 테스트:")
test_cases = [
    ("빅데이터트랙",   "취업/채용", "삼성전자 채용설명회"),
    ("기업경영트랙",   "취업/채용", "삼성전자 채용설명회"),
    ("UX/UI디자인트랙","취업/채용", "삼성전자 채용설명회"),
    ("영미언어정보트랙","공모전/경진대회", "SW 아이디어 경진대회"),
    ("빅데이터트랙",   "공모전/경진대회", "SW 아이디어 경진대회"),
]

for track, category, title in test_cases:
    user_jt = get_user_job_type(track, category)
    print(f"  {track} + [{category}] {title}")
    print(f"  → user_job_type: {user_jt}")

트랙 → 도메인 매핑 테스트:
  빅데이터트랙 → IT
  기업경영트랙 → 경영
  UX/UI디자인트랙 → 디자인
  영미언어정보트랙 → 인문
  발레전공 → 예술
  AI응용학과 → IT
  공공행정트랙 → 행정/공공
  상상력인재학부 (웹공학트랙 선택) → IT
  트랙 미정 → 융합

job_type 매칭 테스트:
  빅데이터트랙 + [취업/채용] 삼성전자 채용설명회
  → user_job_type: IT/개발
  기업경영트랙 + [취업/채용] 삼성전자 채용설명회
  → user_job_type: 경영/금융
  UX/UI디자인트랙 + [취업/채용] 삼성전자 채용설명회
  → user_job_type: 디자인/마케팅
  영미언어정보트랙 + [공모전/경진대회] SW 아이디어 경진대회
  → user_job_type: 인문/사회
  빅데이터트랙 + [공모전/경진대회] SW 아이디어 경진대회
  → user_job_type: 개발/SW


# 6. 테스트

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

# 경로
NOTICES_PATH    = '/content/drive/MyDrive/data_all.json'
MODEL_SAVE_PATH = '/content/drive/MyDrive/two_tower_model.pt'

# 데이터 로드
with open(NOTICES_PATH, encoding='utf-8') as f:
    notices = json.load(f)
print(f"공지 {len(notices)}건 로드 완료")

# SBERT
print("SBERT 로드 중...")
sbert = SentenceTransformer('jhgan/ko-sroberta-multitask')
sbert = sbert.to(device)

# 공지 임베딩
print("공지 임베딩 생성 중...")
notice_texts = [
    f"{n['category']} {n['title']} {n.get('body', '')[:100]}"
    for n in notices
]
notice_embeddings = sbert.encode(
    notice_texts, batch_size=64,
    show_progress_bar=True, device=device, convert_to_numpy=True
)

# notice_score 정규화
scores = np.array([n.get('notice_score', 0) for n in notices])
max_score = scores.max() if scores.max() > 0 else 1
scores_normalized = scores / max_score
notice_id2idx = {n['id']: i for i, n in enumerate(notices)}

# 카테고리 One-Hot
CATEGORIES = ["취업/채용", "학사행정", "장학금", "비교과", "봉사/서포터즈",
              "교육/특강", "공모전/경진대회", "ROTC", "국제교류", "창업", "기숙사/생활관"]
cat2idx = {c: i for i, c in enumerate(CATEGORIES)}
category_onehots = np.array([
    [1.0 if notices[i]['category'] == c else 0.0 for c in CATEGORIES]
    for i in range(len(notices))
], dtype=np.float32)

# 모델 정의
EMBED_DIM  = notice_embeddings.shape[1]
ITEM_DIM = EMBED_DIM + 1
model = TwoTowerModel(EMBED_DIM, ITEM_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()
print("모델 로드 완료!")

# 공지 벡터 계산도 수정 (category onehot 제거)
print("공지 벡터 계산 중...")
all_item_embs = []
with torch.no_grad():
    for i in range(len(notices)):
        item_emb = np.concatenate([
            notice_embeddings[i],
            [scores_normalized[i]]
            # category onehot 제거
        ])
        item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)
        item_vec = model.item_tower(item_tensor)
        all_item_embs.append(item_vec.cpu().numpy())

all_item_embs = np.vstack(all_item_embs)
print(f"완료: {all_item_embs.shape}")

사용 장치: cuda
공지 1921건 로드 완료
SBERT 로드 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


공지 임베딩 생성 중...


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

모델 로드 완료!
공지 벡터 계산 중...
완료: (1921, 128)


In [ ]:
# ================================================================
# 추론 코드 (최종버전 - NO_JOB_TYPE_CATEGORIES 적용)
# ================================================================

MODEL_WEIGHT    = 0.5
CATEGORY_WEIGHT = 0.2
JOB_TYPE_WEIGHT = 0.2
SCORE_WEIGHT    = 0.1

PENALTY_CATEGORIES     = ['국제교류', '봉사/서포터즈', '창업']
NO_JOB_TYPE_CATEGORIES = ['국제교류', '장학금', '학사행정', 'ROTC', '기숙사/생활관', '봉사/서포터즈']
MIN_SCORE_THRESHOLD    = 0.05

def get_job_score(track, notice):
    """job_type 매칭 점수 (제외 카테고리는 0)"""
    if notice.get('category', '') in NO_JOB_TYPE_CATEGORIES:
        return 0.0
    return get_job_type_match_score(track, notice)

# ── 유저 → 공지 추천 함수 ─────────────────────────────────────
def recommend(college, track, year, interests, top_k=10, max_per_category=2):
    user_text   = f"{college} {track} {year} 관심사: {', '.join(interests)}"
    user_emb    = sbert.encode([user_text], convert_to_numpy=True)
    user_tensor = torch.tensor(user_emb, dtype=torch.float).to(device)

    with torch.no_grad():
        user_vec = model.user_tower(user_tensor).cpu().numpy()

    cosine_similarities = np.dot(all_item_embs, user_vec.T).flatten()

    final_results = []
    for n_idx, sim_score in enumerate(cosine_similarities):
        n_score  = scores_normalized[n_idx]
        notice   = notices[n_idx]
        category = notice['category']

        if n_score < MIN_SCORE_THRESHOLD:
            continue

        sim_norm = (sim_score + 1.0) / 2.0

        if category in interests:
            cat_score = 1.0
        elif category in PENALTY_CATEGORIES:
            cat_score = -0.5
        else:
            cat_score = 0.0

        job_score = get_job_score(track, notice)

        final_score = (
            MODEL_WEIGHT    * sim_norm +
            CATEGORY_WEIGHT * cat_score +
            JOB_TYPE_WEIGHT * job_score +
            SCORE_WEIGHT    * n_score
        )

        final_results.append({
            'notice_idx':   n_idx,
            'sim_score':    float(sim_norm),
            'cat_score':    float(cat_score),
            'job_score':    float(job_score),
            'notice_score': float(n_score),
            'final_score':  float(final_score),
            'category':     category,
        })

    final_results.sort(key=lambda x: x['final_score'], reverse=True)

    category_count = {}
    filtered_results = []
    for res in final_results:
        cat = res['category']
        if category_count.get(cat, 0) < max_per_category:
            filtered_results.append(res)
            category_count[cat] = category_count.get(cat, 0) + 1
        if len(filtered_results) == top_k:
            break

    print(f"\n{'='*60}")
    print(f"유저: {college} {track} {year}")
    print(f"관심사: {', '.join(interests)} | 도메인: {get_user_domain(track)}")
    print(f"{'='*60}")
    for rank, res in enumerate(filtered_results):
        n = notices[res['notice_idx']]
        job_types = [t['job_type'] for t in notice_job_types.get(n['id'], [])]
        job_str   = f"[{', '.join(job_types)}]" if job_types else ""
        cat_mark  = " ★" if res['category'] in interests else ""
        job_mark  = " ✓" if res['job_score'] > 0 else ""
        print(f"  {rank+1}. [{n['category']}]{cat_mark}{job_mark} {n['title'][:45]}")
        print(f"     {job_str}")
        print(f"     유사도:{res['sim_score']:.3f} 카테고리:{res['cat_score']:.1f} 직무:{res['job_score']:.1f} | 최종:{res['final_score']:.4f}")


# ── 새 공지 → 유저 추천 함수 ─────────────────────────────────
def recommend_users_for_new_notice(title, category, body="", top_k=5):
    body_preview = body[:100] if body else ""
    notice_text  = f"{category} {title} {body_preview}".strip()
    notice_emb   = sbert.encode([notice_text], convert_to_numpy=True)

    item_emb    = np.concatenate([notice_emb[0], [1.0]])
    item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)

    with torch.no_grad():
        item_vec = model.item_tower(item_tensor).cpu().numpy()

    new_notice = {'id': 'NEW', 'title': title, 'category': category, 'body': body}
    new_notice_types = classify_job_type(new_notice, threshold=0.35)
    notice_job_types['NEW'] = new_notice_types
    new_job_types = [t['job_type'] for t in new_notice_types]

    test_users = [
        {"college": "IT공과대학",             "track": "AI로봇융합트랙",      "year": "1학년", "interests": ["비교과", "학사행정"]},
        {"college": "IT공과대학",             "track": "빅데이터트랙",        "year": "3학년", "interests": ["취업/채용", "공모전/경진대회"]},
        {"college": "IT공과대학",             "track": "모바일소프트웨어트랙", "year": "4학년", "interests": ["취업/채용", "장학금"]},
        {"college": "디자인대학",             "track": "UX/UI디자인트랙",     "year": "2학년", "interests": ["비교과", "봉사/서포터즈"]},
        {"college": "미래융합사회과학대학",    "track": "기업경영트랙",        "year": "4학년", "interests": ["취업/채용", "창업"]},
        {"college": "크리에이티브인문예술대학","track": "영미언어정보트랙",     "year": "2학년", "interests": ["국제교류", "장학금"]},
        {"college": "글로벌인재대학",          "track": "한국언어문화교육학과", "year": "2학년", "interests": ["국제교류", "비교과"]},
        {"college": "미래플러스대학",          "track": "AIㆍ소프트웨어학과",  "year": "3학년", "interests": ["학사행정", "교육/특강"]},
        {"college": "창의융합대학",            "track": "AI응용학과",          "year": "1학년", "interests": ["비교과", "학사행정"]},
        {"college": "IT공과대학",             "track": "웹공학트랙",          "year": "2학년", "interests": ["ROTC", "학사행정"]},
        {"college": "창의융합대학",            "track": "융합보안학과",        "year": "1학년", "interests": ["기숙사/생활관", "장학금"]},
        {"college": "디자인대학",             "track": "패션디자인트랙",      "year": "3학년", "interests": ["봉사/서포터즈", "창업"]},
    ]

    user_scores = []
    with torch.no_grad():
        for user in test_users:
            user_text   = f"{user['college']} {user['track']} {user['year']} 관심사: {', '.join(user['interests'])}"
            user_emb    = sbert.encode([user_text], convert_to_numpy=True)
            user_tensor = torch.tensor(user_emb, dtype=torch.float).to(device)
            user_vec    = model.user_tower(user_tensor).cpu().numpy()

            sim_score = float(np.dot(item_vec, user_vec.T).flatten()[0])
            sim_norm  = (sim_score + 1.0) / 2.0

            if category in user['interests']:
                cat_score = 1.0
            elif category in PENALTY_CATEGORIES:
                cat_score = -0.5
            else:
                cat_score = 0.0

            job_score = get_job_score(user['track'], new_notice)

            final_score = (
                MODEL_WEIGHT    * sim_norm +
                CATEGORY_WEIGHT * cat_score +
                JOB_TYPE_WEIGHT * job_score +
                SCORE_WEIGHT    * 1.0
            )

            user_scores.append({
                'user':        user,
                'sim_score':   sim_norm,
                'job_score':   job_score,
                'final_score': final_score,
            })

    user_scores.sort(key=lambda x: x['final_score'], reverse=True)

    print(f"\n{'='*60}")
    print(f"새 공지: [{category}] {title}")
    if body_preview:
        print(f"본문: {body_preview[:50]}...")
    if new_job_types:
        print(f"공지 job_type: {', '.join(new_job_types)}")
    print(f"{'='*60}")
    for rank, res in enumerate(user_scores[:top_k]):
        u = res['user']
        cat_mark = " ★" if category in u['interests'] else ""
        job_mark = " ✓" if res['job_score'] > 0 else ""
        print(f"  {rank+1}. {u['college']} {u['track']} {u['year']} | 관심사: {', '.join(u['interests'])}{cat_mark}{job_mark}")
        print(f"     유사도:{res['sim_score']:.3f} 직무매칭:{res['job_score']:.1f} | 최종:{res['final_score']:.4f}")

    del notice_job_types['NEW']


# ── 테스트 ────────────────────────────────────────────────────
recommend("IT공과대학", "빅데이터트랙", "3학년", ["취업/채용", "공모전/경진대회"])
recommend("미래융합사회과학대학", "기업경영트랙", "4학년", ["취업/채용", "창업"])

recommend_users_for_new_notice(
    title="2026학년도 하반기 삼성전자 채용설명회 안내",
    category="취업/채용",
    body="삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다."
)
recommend_users_for_new_notice(
    title="2026학년도 국가장학금 2차 신청 안내",
    category="장학금",
    body="한국장학재단에서 2026학년도 국가장학금 2차 신청을 받습니다."
)
recommend_users_for_new_notice(
    title="글로벌 교환학생 파견 프로그램 모집",
    category="국제교류",
    body="2026학년도 2학기 해외 파견 교환학생을 모집합니다. 미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다."
)


유저: IT공과대학 빅데이터트랙 3학년
관심사: 취업/채용, 공모전/경진대회 | 도메인: IT
  1. [취업/채용] ★ ✓ [교육] [가천대 대플] 거점형 [온라인 실시간] 컴활 자격 대비 프로그램 (~4
     [IT/개발, 교내채용]
     유사도:0.998 카테고리:1.0 직무:1.0 | 최종:0.9158
  2. [취업/채용] ★ ✓ [취업행사] 제4회 로봇채용위크 in 용산( 4/29 10:00 )
     [공공/연구, IT/개발]
     유사도:0.994 카테고리:1.0 직무:1.0 | 최종:0.9120
  3. [공모전/경진대회] ★ ✓ [SW중심] 2026학년도 SW중심대학 「에세이 공모전」 안내(재공지)
     [개발/SW, 인문/사회]
     유사도:0.964 카테고리:1.0 직무:1.0 | 최종:0.8871
  4. [공모전/경진대회] ★ ✓ [생성형 AI 활용 경진대회] 제1회 한성 <AX프런티어> 챌린지
     [개발/SW, 창업/마케팅]
     유사도:0.863 카테고리:1.0 직무:1.0 | 최종:0.8439
  5. [비교과] ✓ [SW중심] 제25회 TOPCIT(소프트웨어 역량검정 시험) 정기평가 시행 안내
     [IT교육, 진로/취업]
     유사도:0.994 카테고리:0.0 직무:1.0 | 최종:0.7430
  6. [교육/특강] ✓ [(사)한국교육연구소] AI기반 에너지 컨설턴트 양성과정 교육생 모집
     [AI/IT, 진로/취업]
     유사도:0.978 카테고리:0.0 직무:1.0 | 최종:0.7261
  7. [교육/특강] ✓ [교육][모집 안내(국비)] K-디지털 트레이닝 '빅데이터 디지털금융 전문가' 과
     [AI/IT, 진로/취업]
     유사도:0.990 카테고리:0.0 직무:1.0 | 최종:0.7185
  8. [창업] ✓ [창업멘토링] 2026-1학기 한성 CEO 발굴 창업멘토링 신청 안내
     [일반창업, IT창업]
     유사도:0.980 카테고리:-0.5 직무:1

In [ ]:
import json

CLICKS_PATH = '/content/drive/MyDrive/data_clicks_combined.json'

with open(CLICKS_PATH, encoding='utf-8') as f:
    clicks = json.load(f)

print(f"클릭 데이터 {len(clicks)}개 로드 완료")

# 중복 제거된 유저 풀 생성
seen_texts = set()
unique_users = []

for d in clicks:
    info = d['user']
    user_text = f"{info['college']} {info.get('track', '')} {info['year']} 관심사: {', '.join(info['interests'])}"
    if user_text not in seen_texts:
        seen_texts.add(user_text)
        unique_users.append(info)

print(f"전체 유저: {len(clicks)}명 → 중복 제거 후: {len(unique_users)}명")

def get_notification_targets(title, category, body="", min_score=0.3):
    body_preview = body[:100] if body else ""
    notice_text  = f"{category} {title} {body_preview}".strip()
    notice_emb   = sbert.encode([notice_text], convert_to_numpy=True)

    item_emb    = np.concatenate([notice_emb[0], [1.0]])
    item_tensor = torch.tensor(item_emb, dtype=torch.float).unsqueeze(0).to(device)

    with torch.no_grad():
        item_vec = model.item_tower(item_tensor).cpu().numpy()

    # 새 공지 job_type 분류
    new_notice = {'id': 'NEW', 'title': title, 'category': category, 'body': body}
    new_notice_types = classify_job_type(new_notice, threshold=0.35)
    notice_job_types['NEW'] = new_notice_types
    new_job_types = [t['job_type'] for t in new_notice_types]

    all_users = unique_users

    user_scores = []
    with torch.no_grad():
        for user in all_users:
            user_text   = f"{user['college']} {user.get('track', '')} {user['year']} 관심사: {', '.join(user['interests'])}"
            user_emb    = sbert.encode([user_text], convert_to_numpy=True)
            user_tensor = torch.tensor(user_emb, dtype=torch.float).to(device)
            user_vec    = model.user_tower(user_tensor).cpu().numpy()

            sim_score = float(np.dot(item_vec, user_vec.T).flatten()[0])
            sim_norm  = (sim_score + 1.0) / 2.0

            if category in user['interests']:
                cat_score = 1.0
            elif category in PENALTY_CATEGORIES:
                cat_score = -0.5
            else:
                cat_score = 0.0

            job_score = get_job_score(user.get('track', ''), new_notice)

            final_score = (
                MODEL_WEIGHT    * sim_norm +
                CATEGORY_WEIGHT * cat_score +
                JOB_TYPE_WEIGHT * job_score +
                SCORE_WEIGHT    * 1.0
            )

            user_scores.append({
                'user':        user,
                'final_score': final_score,
                'sim_score':   sim_norm,
                'cat_score':   cat_score,
                'job_score':   job_score,
            })

    user_scores.sort(key=lambda x: x['final_score'], reverse=True)

    scores     = np.array([u['final_score'] for u in user_scores])
    mean_score = scores.mean()
    std_score  = scores.std()
    cut_score  = mean_score + std_score

    targets = [u for u in user_scores if u['final_score'] >= max(cut_score, min_score)]

    print(f"\n{'='*60}")
    print(f"새 공지: [{category}] {title}")
    if new_job_types:
        print(f"공지 job_type: {', '.join(new_job_types)}")
    print(f"전체 유저: {len(user_scores)}명 | 알림 대상: {len(targets)}명 ({len(targets)/len(user_scores)*100:.1f}%)")
    print(f"{'='*60}")
    print(f"평균: {mean_score:.4f} | 표준편차: {std_score:.4f} | 컷 기준: {cut_score:.4f}")

    print(f"\n알림 대상 Top 10:")
    for rank, res in enumerate(targets[:10]):
        u = res['user']
        cat_mark = " ★" if category in u['interests'] else ""
        job_mark = " ✓" if res['job_score'] > 0 else ""
        print(f"  {rank+1}. {u['college']} {u.get('track','')} {u['year']} | 관심사: {', '.join(u['interests'])}{cat_mark}{job_mark}")
        print(f"     유사도:{res['sim_score']:.3f} 카테고리:{res['cat_score']:.1f} 직무:{res['job_score']:.1f} | 최종:{res['final_score']:.4f}")

    print(f"\n점수 구간별 유저 수:")
    bins = [1.2, 1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
    for i in range(len(bins)-1):
        count = sum(1 for s in scores if bins[i+1] <= s < bins[i])
        bar = "█" * (count // 5)
        print(f"  {bins[i+1]:.1f}~{bins[i]:.1f}: {count:3d}명 {bar}")

    del notice_job_types['NEW']
    return targets

# 11개 카테고리 전체 테스트
test_notices = [
    {
        "title": "2026학년도 하반기 삼성전자 채용설명회 안내",
        "category": "취업/채용",
        "body": "삼성전자 DS부문에서 소프트웨어, 하드웨어, AI 직군 신입사원을 모집합니다."
    },
    {
        "title": "2026학년도 1학기 수강신청 안내",
        "category": "학사행정",
        "body": "2026학년도 1학기 수강신청 기간 및 방법을 안내합니다."
    },
    {
        "title": "2026학년도 국가장학금 2차 신청 안내",
        "category": "장학금",
        "body": "한국장학재단에서 2026학년도 국가장학금 2차 신청을 받습니다. 소득분위 8구간 이하 재학생이라면 누구나 신청 가능하며, 최대 700만원까지 지원됩니다."
    },
    {
        "title": "2026학년도 1학기 비교과 프로그램 참가자 모집",
        "category": "비교과",
        "body": "다양한 비교과 활동 프로그램 참가자를 모집합니다. 비교과 포인트가 지급됩니다."
    },
    {
        "title": "2026학년도 대학생 봉사단 모집",
        "category": "봉사/서포터즈",
        "body": "지역사회 봉사활동 참여자를 모집합니다. 봉사시간이 인정됩니다."
    },
    {
        "title": "AI 활용 디지털 역량 강화 특강",
        "category": "교육/특강",
        "body": "생성형 AI 실습 교육 프로그램입니다. AI 활용 능력을 키울 수 있습니다."
    },
    {
        "title": "2026학년도 SW 아이디어 경진대회",
        "category": "공모전/경진대회",
        "body": "소프트웨어 관련 아이디어 공모전입니다. 우수작에는 상금이 수여됩니다."
    },
    {
        "title": "2026년 학군사관 ROTC 67기 모집 안내",
        "category": "ROTC",
        "body": "한성대학교 학군단에서 ROTC 67기 후보생을 모집합니다."
    },
    {
        "title": "글로벌 교환학생 파견 프로그램 모집",
        "category": "국제교류",
        "body": "2026학년도 2학기 해외 파견 교환학생을 모집합니다. 미국, 일본, 유럽 등 30개국 50개 대학에 파견됩니다."
    },
    {
        "title": "2026학년도 창업 아이디어 경진대회",
        "category": "창업",
        "body": "예비창업가 대상 창업 지원 프로그램입니다. 우수팀에게 창업 지원금이 제공됩니다."
    },
    {
        "title": "2026학년도 2학기 기숙사 입사 신청 안내",
        "category": "기숙사/생활관",
        "body": "2026학년도 2학기 기숙사 입사 신청을 받습니다. 신청 기간 및 방법을 확인하세요."
    },
]

for notice in test_notices:
    _ = get_notification_targets(
        title=notice['title'],
        category=notice['category'],
        body=notice['body'],
    )
    print()

클릭 데이터 970개 로드 완료
전체 유저: 970명 → 중복 제거 후: 814명

새 공지: [취업/채용] 2026학년도 하반기 삼성전자 채용설명회 안내
공지 job_type: IT/개발
전체 유저: 814명 | 알림 대상: 152명 (18.7%)
평균: 0.5661 | 표준편차: 0.1789 | 컷 기준: 0.7450

알림 대상 Top 10:
  1. IT공과대학 AI로봇융합트랙 3학년 | 관심사: 취업/채용, 공모전/경진대회 ★ ✓
     유사도:0.990 카테고리:1.0 직무:1.0 | 최종:0.9949
  2. IT공과대학 시스템반도체트랙 4학년 | 관심사: 취업/채용, 장학금 ★ ✓
     유사도:0.989 카테고리:1.0 직무:1.0 | 최종:0.9946
  3. IT공과대학 산업공학트랙 4학년 | 관심사: 취업/채용, 장학금 ★ ✓
     유사도:0.989 카테고리:1.0 직무:1.0 | 최종:0.9943
  4. IT공과대학 전자트랙 4학년 | 관심사: 취업/채용, 장학금 ★ ✓
     유사도:0.988 카테고리:1.0 직무:1.0 | 최종:0.9939
  5. IT공과대학 기계시스템디자인트랙 3학년 | 관심사: 장학금, 취업/채용 ★ ✓
     유사도:0.988 카테고리:1.0 직무:1.0 | 최종:0.9938
  6. IT공과대학 산업공학트랙 4학년 | 관심사: 창업, 취업/채용 ★ ✓
     유사도:0.987 카테고리:1.0 직무:1.0 | 최종:0.9935
  7. IT공과대학 시스템반도체트랙 4학년 | 관심사: 공모전/경진대회, 취업/채용 ★ ✓
     유사도:0.985 카테고리:1.0 직무:1.0 | 최종:0.9926
  8. IT공과대학 시스템반도체트랙 4학년 | 관심사: 취업/채용, 공모전/경진대회, 장학금 ★ ✓
     유사도:0.985 카테고리:1.0 직무:1.0 | 최종:0.9923
  9. IT공과대학 빅데이터트랙 4학년 | 관심사: 창업, 취업/채용 ★ ✓
     유사도:0.984 카테고리:1.0 직무:

#7. 평가지표

In [ ]:
from collections import defaultdict
import numpy as np
import torch

MODEL_WEIGHT    = 0.5
JOB_TYPE_WEIGHT = 0.4
SCORE_WEIGHT    = 0.1


def evaluate_hitrate_only(clicks_data, K_list=[10, 30, 50]):
    user_positives  = defaultdict(list)
    user_embeddings = {}
    user_interests  = {}
    user_tracks     = {}

    for user_data in clicks_data:
        info = user_data["user"]
        user_text = f"{info['college']} {info.get('track', '')} {info['year']} 관심사: {', '.join(info['interests'])}"

        if user_text not in user_emb_dict:
            continue

        for pos_id in user_data["clicks"].get("positive", []):
            if pos_id in notice_id2idx:
                user_positives[user_text].append(notice_id2idx[pos_id])
                user_embeddings[user_text] = user_emb_dict[user_text]
                user_interests[user_text] = info.get("interests", [])
                user_tracks[user_text] = info.get("track", "")

    hitrate = {k: [] for k in K_list}
    candidate_sizes = []
    eval_positive_counts = []

    all_indices = list(range(len(notices)))

    with torch.no_grad():
        for uid, pos_indices in user_positives.items():
            user_emb = user_embeddings[uid]
            interests = user_interests[uid]
            track = user_tracks[uid]

            user_tensor = torch.tensor(user_emb, dtype=torch.float).unsqueeze(0).to(device)
            user_vec = model.user_tower(user_tensor).cpu().numpy()

            model_scores = np.dot(all_item_embs, user_vec.T).flatten()

            candidate_idx = [
                i for i, n in enumerate(notices)
                if n.get("category", "") in interests
            ]

            if len(candidate_idx) == 0:
                continue

            filtered_pos = set(pos_indices) & set(candidate_idx)

            if len(filtered_pos) == 0:
                continue

            candidate_sizes.append(len(candidate_idx))
            eval_positive_counts.append(len(filtered_pos))

            final_scores = []

            for n_idx in candidate_idx:
                notice = notices[n_idx]
                n_score = scores_normalized[n_idx]
                sim_score = model_scores[n_idx]
                sim_norm = (sim_score + 1.0) / 2.0

                job_score = get_job_score(track, notice)

                if job_score <= 0:
                    job_score = JOB_PENALTY

                final_score = (
                    MODEL_WEIGHT * sim_norm
                    + JOB_TYPE_WEIGHT * job_score
                    + SCORE_WEIGHT * n_score
                )

                final_scores.append(final_score)

            final_scores = np.array(final_scores)
            ranked_local = np.argsort(final_scores)[::-1]
            ranked = [candidate_idx[i] for i in ranked_local]

            for K in K_list:
                top_k = set(ranked[:K])
                hit_count = len(filtered_pos & top_k)
                hitrate[K].append(1.0 if hit_count > 0 else 0.0)

    print(f"\n{'='*55}")
    print(f"{'Two-Tower + job_type HitRate 평가':^55}")
    print(f"{'Category Filter + Job Soft Filter':^55}")
    print(f"{'='*55}")
    print(f"평균 후보 공지 수: {np.mean(candidate_sizes):.0f}개")
    print(f"평균 평가 positive 수: {np.mean(eval_positive_counts):.2f}개")
    print(f"평가 유저 수: {len(hitrate[K_list[0]])}명")
    print(f"평가 positive 수: {sum(eval_positive_counts)}개")
    print(f"Weight: MODEL={MODEL_WEIGHT}, JOB={JOB_TYPE_WEIGHT}, SCORE={SCORE_WEIGHT}")
    print(f"{'-'*55}")
    print(f"{'지표':<15} {'@10':>10} {'@30':>10} {'@50':>10}")
    print(f"{'-'*55}")

    values = [np.mean(hitrate[k]) if len(hitrate[k]) > 0 else 0.0 for k in K_list]
    print(f"{'HitRate':<15} {values[0]:>10.4f} {values[1]:>10.4f} {values[2]:>10.4f}")

    print(f"{'='*55}")

    return {
        f"HitRate@{k}": float(np.mean(hitrate[k])) if len(hitrate[k]) > 0 else 0.0
        for k in K_list
    }

hitrate_summary = evaluate_hitrate_only(test_clicks)


            Two-Tower + job_type HitRate 평가            
           Category Filter + Job Soft Filter           
평균 후보 공지 수: 493개
평균 평가 positive 수: 6.50개
평가 유저 수: 133명
평가 positive 수: 864개
Weight: MODEL=0.5, JOB=0.4, SCORE=0.1
-------------------------------------------------------
지표                     @10        @30        @50
-------------------------------------------------------
HitRate             0.2857     0.5639     0.6391


## 성능이 더욱 좋아진 것을 확인할 수 있다.